# 🚀 LAB GUIDE — PRODUCTION-GRADE GRAPHRAG VS FLAT RAG

**Thời lượng:** 120 phút  
**Môi trường:** Windows + `.venv` + JupyterLab + Neo4j Docker  
**Dữ liệu:** HackerNoon Tech Company News Data Dump (bản thu gọn do giảng viên cung cấp)  
**Công cụ:** Học viên được dùng AI Coding Agent, nhưng phải tự thiết kế, kiểm thử và giải thích logic.

## 🎯 Mục tiêu
1. Xây dựng Hybrid GraphRAG end-to-end.
2. Xử lý Coreference Resolution, Entity Resolution và Super-node Mitigation.
3. Bulk insert bằng `UNWIND`, không insert từng row.
4. So sánh Flat RAG và GraphRAG bằng Golden Dataset + LLM-as-a-Judge.
5. Đo quality, latency và token usage.
6. Giải thích kiến trúc và failure modes.

> Notebook là **reference lab guide**: có code khung chạy được nhưng vẫn yêu cầu học viên thay prompt/threshold/retrieval policy và thuyết minh lựa chọn.

**Bài nộp Day19 — Vũ Duy Diệp:** code pipeline nằm trong các cell của notebook. Kết quả: hai CSV trong outputs/ và bốn báo cáo trong reports/.


## ⏳ Timeline

| Phút | Nội dung |
|---|---|
| 00–15 | Setup, load, dedup, chunk, coreference |
| 15–45 | NER/RE, entity resolution, Neo4j bulk insert |
| 45–75 | Flat RAG, graph traversal, hybrid retrieval |
| 75–105 | Golden Dataset, LLM-as-a-Judge, comparison |
| 105–120 | Failure-mode tests, bonus, export, thuyết minh |

### Scale guard
Trong lab 2 giờ, không nên gửi toàn bộ 350MB qua LLM. Mặc định dùng subset:
- `LAB_MAX_ARTICLES = 1500`
- `LAB_MAX_CHUNKS = 3000`
- `EXTRACTION_MAX_CHUNKS = 400`

Kiến trúc phải scale được; volume trong giờ lab chỉ dùng để chứng minh pipeline.

# PHẦN 1 — SETUP & PREPROCESSING

Chạy Jupyter từ thư mục repo, dùng kernel của `.venv`. Khi dựng lại môi trường: `pip install -r requirements.txt jupyterlab ipykernel`.
Secrets được đọc từ `.env` local hoặc Colab Secrets; không lưu key/password trong notebook.
Neo4j local: `bolt://localhost:7687`. Lần chạy bài nộp dùng Groq cho generator và Judge, model `openai/gpt-oss-120b`.
Model Llama mặc định đã trả 404 khi kiểm tra tài khoản trong phiên thực hiện; cấu hình thực dùng được ghi ở output cell config.
Đặt trong `.env`: GROQ_MODEL=openai/gpt-oss-120b, JUDGE_PROVIDER=groq, JUDGE_MODEL=openai/gpt-oss-120b.
Mặc định extraction 200 chunks, batches 8; vẫn nằm trong giới hạn 400. Không cần OpenAI/Gemini key khi dùng Groq cho Judge.
Code dưới đây chạy độc lập với các module và script hỗ trợ ngoài notebook. Cache/logs sinh ở local để resume và không thuộc bộ file nộp.


In [1]:
# 1.1 — Local environment: install only when recreating .venv
import sys, importlib.util
required_modules = ['neo4j', 'pandas', 'numpy', 'sentence_transformers', 'faiss', 'groq', 'openai', 'dotenv']
missing_modules = [name for name in required_modules if importlib.util.find_spec(name) is None]
print('Kernel:', sys.executable)
print('Missing modules:', missing_modules)
if missing_modules:
    raise RuntimeError('Install requirements.txt plus jupyterlab/ipykernel in this kernel environment.')


Kernel: D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703\.venv\Scripts\python.exe
Missing modules: []


In [2]:
"""Checkpointing, provenance mapping and local configuration for Lab 19."""
import hashlib
import json
import math
import re
import subprocess
import unicodedata
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import dotenv_values

ROOT = Path.cwd().resolve()
OUTPUTS = ROOT / 'outputs'
CACHE = OUTPUTS / '.cache'
PROMPT_VERSION = 'lab19-v1'
JUDGE_KEY_NAMES = {'openai': 'OPENAI_API_KEY', 'groq': 'GROQ_API_KEY', 'gemini': 'GEMINI_API_KEY'}
OUTPUTS.mkdir(exist_ok=True)
CACHE.mkdir(exist_ok=True)


def is_configured(value):
    value = str(value or '').strip()
    return bool(value) and not any(x in value for x in ('...', '<your', 'your-'))


def fingerprint(value):
    if isinstance(value, pd.DataFrame):
        value = value.fillna('').to_dict('records')
    return hashlib.sha256(json.dumps(value, sort_keys=True, ensure_ascii=False,
                                     default=str).encode('utf-8')).hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    default=str, allow_nan=False), encoding='utf-8')
    temporary.replace(path)


def cache_path(stage, payload):
    return CACHE / f'{stage}_{fingerprint(payload)}.json'


def cached_json(stage, payload, callback):
    path = cache_path(stage, payload)
    if path.exists():
        return json.loads(path.read_text(encoding='utf-8'))
    value = callback()
    write_json(path, value)
    return value





def norm_text(value):
    if value is None or isinstance(value, float) and math.isnan(value):
        return ''
    return re.sub(r'\s+', ' ', unicodedata.normalize('NFKC', str(value))).strip()


def column(df, candidates, required=False):
    lookup = {re.sub(r'[\s_-]+', '', str(c).lower()): c for c in df.columns}
    for name in candidates:
        key = re.sub(r'[\s_-]+', '', name.lower())
        if key in lookup:
            return lookup[key]
    if required:
        raise KeyError(f'Missing a column in {candidates}')
    return None


def source_matches(raw, detailed):
    """Match source identity, not a possibly shifted streaming row number."""
    title_col = column(raw, ['title', 'headline'], required=True)
    date_col = column(raw, ['published_date', 'date', 'published_at', 'created_at', 'timestamp'])
    titles = raw[title_col].fillna('').map(lambda x: norm_text(x).casefold())
    dates = pd.to_datetime(raw[date_col], utc=True, errors='coerce').dt.strftime('%Y-%m-%d') if date_col else pd.Series('', index=raw.index)
    url_columns = [c for c in raw.columns if any(word in str(c).lower() for word in ('url', 'link'))]
    records = []
    for q in detailed.itertuples(index=False):
        row_ids = json.loads(q.evidence_row_ids_0based)
        urls = json.loads(q.evidence_urls_json)
        sources = re.findall(r'row\s+(\d+)\s+\(([^)]+)\):\s*(.*?)(?=\s+\|\s+row\s+\d+|$)', str(q.reference_evidence))
        references = {int(row_id): (date[:10], title.strip()) for row_id, date, title in sources}
        for position, original_row_id in enumerate(row_ids):
            url = urls[position] if position < len(urls) else ''
            expected_date, expected_title = references.get(original_row_id, ('', ''))
            mask = pd.Series(False, index=raw.index)
            if url:
                for field in url_columns:
                    mask |= raw[field].fillna('').astype(str).str.rstrip('/').eq(url.rstrip('/'))
            mode = 'url'
            if not mask.any() and expected_title:
                mask = titles.eq(norm_text(expected_title).casefold())
                if date_col and expected_date:
                    mask &= dates.eq(expected_date)
                mode = 'title_date' if date_col else 'title_only'
            matched = raw.index[mask].tolist()
            records.append({'id': q.id, 'original_row_id': original_row_id,
                            'matched_source_row_ids': json.dumps(matched),
                            'matched': bool(matched), 'match_mode': mode if matched else 'missing',
                            'expected_title': expected_title, 'expected_url': url})
    return pd.DataFrame(records)


def choose_golden(detailed, matches, count=5):
    coverage = matches.groupby('id')['matched'].all()
    eligible = detailed[detailed.id.isin(coverage[coverage].index)].copy()
    selected = []
    for group, quota in [('factoid', 1), ('multi-hop', 2), ('cross-doc', 2)]:
        selected.extend(eligible[eligible.group.eq(group)].sort_values('id').head(quota).id.tolist())
    if len(selected) < 5:
        raise ValueError('Need verified source coverage for 1 factoid, 2 multi-hop and 2 cross-doc questions.')
    selected.extend(eligible[~eligible.id.isin(selected)].sort_values('id').head(max(0, count - 5)).id.tolist())
    result = eligible[eligible.id.isin(selected)].sort_values('id').copy()
    result.to_csv(ROOT / 'data' / 'graphrag_golden_lab.csv', index=False)
    return result


def curated_rows(matches, golden):
    return sorted({row_id for value in matches[matches.id.isin(golden.id)].matched_source_row_ids
                   for row_id in json.loads(value)})


def prepare_corpus():
    """Freeze source-selected inputs before evaluation."""
    global LAB_NAMESPACE
    raw = load_news(DATA_PATH).reset_index(drop=True)
    detailed = pd.read_csv(ROOT / 'data' / 'graphrag_golden_50_first5000_detailed.csv').fillna('')
    matches = source_matches(raw, detailed)
    matches.to_csv(OUTPUTS / 'golden_source_matches.csv', index=False)
    golden = choose_golden(detailed, matches, count=GOLDEN_COUNT)
    priority_rows = curated_rows(matches, golden)
    news = standardize_news(raw, priority_rows=priority_rows)
    priority_article_ids = set()
    for row in news.itertuples(index=False):
        if set(json.loads(row.source_row_ids_json)) & set(priority_rows):
            priority_article_ids.add(row.article_id)
    ordered = pd.concat([news[news.article_id.isin(priority_article_ids)],
                         news[~news.article_id.isin(priority_article_ids)]], ignore_index=True)
    chunks = build_chunks(ordered)
    extraction = chunks.head(EXTRACTION_TARGET_CHUNKS).copy()
    # Do not put references, expected relations or answers into extraction/chunk inputs.
    rows = []
    for q in golden.itertuples(index=False):
        matched = matches[matches.id.eq(q.id)]
        required = {rid for value in matched.matched_source_row_ids for rid in json.loads(value)}
        def coverage(frame):
            available = {rid for value in frame.source_row_ids_json for rid in json.loads(value)}
            return required <= available
        rows.append({'id': q.id, 'group': q.group, 'corpus_covered': True,
                     'vector_covered': coverage(chunks), 'extraction_covered': coverage(extraction)})
    coverage_df = pd.DataFrame(rows)
    coverage_df.to_csv(OUTPUTS / 'golden_coverage.csv', index=False)
    if not coverage_df.vector_covered.all():
        raise ValueError('Chunk cap removed required Golden sources; adjust sampling before evaluation.')
    chunks.to_csv(OUTPUTS / 'raw_chunks.csv', index=False)
    chunks.drop(columns=['text']).to_csv(OUTPUTS / 'chunk_manifest.csv', index=False)
    corpus_manifest = {
        'sample_policy': 'coverage-selected demonstration using source identity, not answer contents',
        'corpus_fingerprint': fingerprint(raw), 'chunks_fingerprint': fingerprint(chunks),
        'extraction_fingerprint': fingerprint(extraction), 'golden_ids': golden.id.tolist(),
        'priority_source_row_ids': priority_rows, 'article_ids': ordered.article_id.tolist(),
        'extraction_chunk_ids': extraction.chunk_id.tolist(), 'raw_rows': len(raw),
        'articles': len(ordered), 'chunks': len(chunks), 'extraction_chunks': len(extraction),
        'text_kind': pd.read_csv(OUTPUTS / 'preprocessing_stats.csv').iloc[0].text_kind,
        'seed': SEED, 'frozen_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    LAB_NAMESPACE = 'day19-' + fingerprint({
        'chunks': corpus_manifest['chunks_fingerprint'],
        'extraction': corpus_manifest['extraction_fingerprint'],
        'model': GROQ_MODEL,
        'runtime_version': 'fc3502932347b75ee540756993a1f65ff8d4465e4da7689c205f9a11c273f4ba',
    })[:16]
    corpus_manifest['graph_namespace'] = LAB_NAMESPACE
    write_json(OUTPUTS / 'corpus_manifest.json', corpus_manifest)
    return ordered, chunks, extraction, golden, coverage_df

"""Lab 19 runtime; notebook cells call these functions in section order."""
from IPython.display import display
from dotenv import load_dotenv, dotenv_values
import logging
import math
from datetime import datetime, timezone

#@title 1.2 — Imports & config
import os, re, json, time, random, hashlib, unicodedata
from pathlib import Path
from collections import defaultdict, Counter, deque
from difflib import SequenceMatcher

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from neo4j import GraphDatabase
from sentence_transformers import SentenceTransformer
import faiss

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)
load_dotenv(ROOT / '.env')
LOCAL_CONFIG = dotenv_values(ROOT / '.env')
USAGE_LOG = []
LAB_NAMESPACE = 'day19-tech-news-v1'
GOLDEN_COUNT = int(os.getenv('LAB_GOLDEN_COUNT', '5'))


def safe_error(error):
    message = str(error)
    for name in ['NEO4J_PASSWORD', 'GROQ_API_KEY', 'OPENAI_API_KEY', 'GEMINI_API_KEY', 'HF_TOKEN']:
        for value in (os.getenv(name, ''), LOCAL_CONFIG.get(name, '')):
            if value:
                message = message.replace(value, '[REDACTED]')
    return f'{type(error).__name__}: {message[:600]}'

def get_secret(name, default=None):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value is not None:
            return value
    except Exception:
        pass
    value = LOCAL_CONFIG.get(name)
    if not is_configured(value):
        value = os.environ.get(name, default)
    return value if is_configured(value) else default

NEO4J_URI = get_secret("NEO4J_URI", "")
NEO4J_USER = get_secret("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = get_secret("NEO4J_PASSWORD", "")
NEO4J_DATABASE = get_secret("NEO4J_DATABASE", "neo4j")

GROQ_API_KEY = get_secret("GROQ_API_KEY", "")
GROQ_MODEL = get_secret("GROQ_MODEL", "")
GROQ_REASONING_EFFORT = get_secret('GROQ_REASONING_EFFORT', 'low')

JUDGE_PROVIDER = get_secret("JUDGE_PROVIDER", "groq").lower()
JUDGE_MODEL = get_secret("JUDGE_MODEL", "")
OPENAI_API_KEY = get_secret("OPENAI_API_KEY", "")
GEMINI_API_KEY = get_secret("GEMINI_API_KEY", "")
HF_TOKEN = get_secret("HF_TOKEN", "")

DATA_PATH = Path(os.getenv('LAB_DATA_PATH', str(ROOT / 'data' / 'hackernoon_subset.csv')))
GOLDEN_PATH = ROOT / 'data' / 'graphrag_golden_lab.csv'
CHECKPOINT = OUTPUTS / 'graphrag_eval_checkpoint.csv'
LAB_MAX_ARTICLES = 1500
LAB_MAX_CHUNKS = 3000
EXTRACTION_MAX_CHUNKS = 400
EXTRACTION_TARGET_CHUNKS = int(get_secret('LAB_EXTRACTION_CHUNKS', 200))
COREF_BATCH_SIZE = int(get_secret('LAB_COREF_BATCH_SIZE', 8))
EXTRACTION_BATCH_SIZE = int(get_secret('LAB_EXTRACTION_BATCH_SIZE', 8))
if not 1 <= EXTRACTION_TARGET_CHUNKS <= EXTRACTION_MAX_CHUNKS:
    raise ValueError('LAB_EXTRACTION_CHUNKS must be between 1 and 400.')
if not 1 <= COREF_BATCH_SIZE <= 16 or not 1 <= EXTRACTION_BATCH_SIZE <= 16:
    raise ValueError('LLM batch sizes must be between 1 and 16.')
CHUNK_WORDS = 220
CHUNK_OVERLAP_WORDS = 40

data_ready = graph_ready = retrieval_ready = False
eval_results_df = None
print('Repo:', ROOT)
print('Generator:', GROQ_MODEL, '| Judge:', JUDGE_PROVIDER, JUDGE_MODEL)
print('Scale guard:', LAB_MAX_ARTICLES, LAB_MAX_CHUNKS, EXTRACTION_MAX_CHUNKS)
print('Selected extraction chunks:', EXTRACTION_TARGET_CHUNKS)


D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Repo: D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703
Generator: openai/gpt-oss-120b | Judge: groq openai/gpt-oss-120b
Scale guard: 1500 3000 400
Selected extraction chunks: 200


## 1.3 — Corpus nguồn và streaming có giới hạn

Dùng `data/hackernoon_subset.csv` nếu đã có; nếu chưa có, cell bên dưới stream tối đa 5.000 rows/300 MiB từ revision đã ghi nhận.
Dataset cần HF_TOKEN và tài khoản đã chấp nhận điều kiện truy cập. Không commit corpus tải xuống.
Khớp nguồn Golden bằng URL hoặc title/date; không giả định row index giữa các lần tải giống nhau.
Scale guard: 1.500 articles, 3.000 chunks, 400 extraction chunks; lần chạy này dùng 1.500/1.500/200.


In [3]:
# 1.3 — Reuse the original corpus, or download the pinned HackerNoon stream.
def download_news_subset(target, revision='cc6144ccce683dcebb6e63f9a50ca084544af1c0'):
    import csv
    from datasets import load_dataset
    if not HF_TOKEN:
        raise RuntimeError('Configure HF_TOKEN and accept the dataset access conditions.')
    target.parent.mkdir(parents=True, exist_ok=True)
    temporary = target.with_suffix('.csv.part')
    stream = load_dataset('HackerNoon/tech-company-news-data-dump', split='train',
                          streaming=True, token=HF_TOKEN, revision=revision)
    count = 0
    with temporary.open('w', encoding='utf-8', newline='') as handle:
        writer = None
        for row in stream:
            if writer is None:
                columns = list(row)
                writer = csv.DictWriter(handle, fieldnames=columns, extrasaction='ignore')
                writer.writeheader()
            writer.writerow({key: json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else value
                             for key, value in row.items()})
            count += 1
            if count % 250 == 0:
                handle.flush()
            if count >= 5000 or temporary.stat().st_size >= 300 * 1024 * 1024:
                break
    if not count:
        raise RuntimeError('Dataset stream was empty.')
    temporary.replace(target)
    print('Downloaded rows:', count, '| revision:', revision)

if not DATA_PATH.exists():
    try:
        download_news_subset(DATA_PATH)
    except Exception as error:
        raise RuntimeError(safe_error(error)) from None
print('Using corpus:', DATA_PATH)


Using corpus: D:\VinUniversity\K4-DAY19-VuDuyDiep-2A202602703\data\hackernoon_subset.csv


In [4]:
#@title 1.4 — Neo4j connection + schema
driver = None

def connect_neo4j():
    global driver
    if not NEO4J_URI or not NEO4J_PASSWORD:
        raise ValueError("Thiếu Neo4j secrets.")
    driver = GraphDatabase.driver(
        NEO4J_URI,
        auth=(NEO4J_USER, NEO4J_PASSWORD),
    )
    driver.verify_connectivity()
    print('Neo4j connected.')

def run_cypher(query, **params):
    if driver is None:
        raise RuntimeError("Hãy chạy connect_neo4j() trước.")
    with driver.session(database=NEO4J_DATABASE) as session:
        result = session.run(query, namespace=LAB_NAMESPACE, **params)
        rows = [r.data() for r in result]
        result.consume()
    return rows

def setup_graph_schema():
    for stmt in [
        """
        CREATE CONSTRAINT entity_id IF NOT EXISTS
        FOR (n:Entity) REQUIRE n.id IS UNIQUE
        """,
        """
        CREATE INDEX entity_name_norm IF NOT EXISTS
        FOR (n:Entity) ON (n.name_norm)
        """,
        """
        CREATE INDEX company_name_norm IF NOT EXISTS
        FOR (n:Company) ON (n.name_norm)
        """,
        """
        CREATE INDEX person_name_norm IF NOT EXISTS
        FOR (n:Person) ON (n.name_norm)
        """,
        """
        CREATE INDEX technology_name_norm IF NOT EXISTS
        FOR (n:Technology) ON (n.name_norm)
        """,
    ]:
        run_cypher(stmt)
    print('Schema ready.')

# connect_neo4j()
# setup_graph_schema()

def preflight(check_llm=True):
    statuses = []
    required = {'NEO4J_URI': NEO4J_URI, 'NEO4J_PASSWORD': NEO4J_PASSWORD,
                'GROQ_API_KEY': GROQ_API_KEY, 'GROQ_MODEL': GROQ_MODEL, 'JUDGE_MODEL': JUDGE_MODEL}
    judge_key_name = JUDGE_KEY_NAMES.get(JUDGE_PROVIDER)
    judge_key = globals()[judge_key_name] if judge_key_name else ''
    statuses.append({'check': 'JUDGE_PROVIDER', 'status': 'configured' if judge_key_name else 'failed',
                     'detail': JUDGE_PROVIDER if judge_key_name else 'Expected openai, groq or gemini'})
    if judge_key_name:
        required[judge_key_name] = judge_key
    for name, value in required.items():
        statuses.append({'check': name, 'status': 'configured' if is_configured(value) else 'missing_or_placeholder'})
    statuses.append({'check': 'corpus', 'status': 'present' if DATA_PATH.exists() else 'missing'})
    statuses.append({'check': 'HF_TOKEN', 'status': 'configured' if HF_TOKEN else 'needed_for_gated_download'})
    try:
        connect_neo4j()
        assert run_cypher('RETURN 1 AS ok')[0]['ok'] == 1
        existing = run_cypher('MATCH (n) RETURN count(n) AS nodes')[0]['nodes']
        statuses.append({'check': 'neo4j_driver', 'status': 'passed', 'detail': f'{existing} existing nodes; lab queries are namespaced'})
        setup_graph_schema()
    except Exception as error:
        statuses.append({'check': 'neo4j_driver', 'status': 'failed', 'detail': safe_error(error)})
    if check_llm and GROQ_API_KEY and GROQ_MODEL:
        try:
            groq_json('Return strict JSON.', 'Return {"ok": true}', purpose='preflight')
            statuses.append({'check': 'groq_request', 'status': 'passed'})
        except Exception as error:
            statuses.append({'check': 'groq_request', 'status': 'failed', 'detail': safe_error(error)})
    if check_llm and JUDGE_MODEL and is_configured(judge_key):
        try:
            probe = judge_json('Return strict JSON.', 'Return {"ok": true}', purpose='preflight')
            if probe.get('ok') is not True:
                raise ValueError('Judge probe did not return ok=true.')
            statuses.append({'check': 'judge_request', 'status': 'passed'})
        except Exception as error:
            statuses.append({'check': 'judge_request', 'status': 'failed', 'detail': safe_error(error)})
    result = pd.DataFrame(statuses).fillna('')
    result.to_csv(OUTPUTS / 'preflight_status.csv', index=False)
    return result

# 1.4 — Driver, schema, secret status and minimal provider probes
preflight_df = preflight(check_llm=False)
display(preflight_df)
neo4j_ready = preflight_df.loc[preflight_df.check.eq('neo4j_driver'), 'status'].eq('passed').all()
llm_ready = not preflight_df.status.isin(['missing_or_placeholder', 'failed']).any()
print('Neo4j ready:', neo4j_ready, '| LLM ready:', llm_ready)


Neo4j connected.
Schema ready.


,check,status,detail
0,JUDGE_PROVIDER,configured,groq
1,NEO4J_URI,configured,
2,NEO4J_PASSWORD,configured,
3,GROQ_API_KEY,configured,
4,GROQ_MODEL,configured,
5,JUDGE_MODEL,configured,
6,corpus,present,
7,HF_TOKEN,configured,
8,neo4j_driver,passed,194 existing nodes; lab queries are namespaced


Neo4j ready:

 True | LLM ready: True


In [5]:
#@title 1.5 — Loader + exact dedup + chunking
def norm_space(x):
    return norm_text(x)

def sha1(x):
    return hashlib.sha1(str(x).encode("utf-8", errors="ignore")).hexdigest()

def pick_col(df, candidates, required=True):
    return column(df, candidates, required=required)

def load_news(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".jsonl", ".ndjson"}:
        return pd.read_json(path, lines=True)
    if path.suffix.lower() == ".json":
        return pd.read_json(path)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported: {path.suffix}")

def standardize_news(raw, priority_rows=None):
    text_col = pick_col(raw, ["text", "content", "article", "body", "story"], required=False)
    description_col = pick_col(raw, ['description', 'summary', 'snippet'], required=False)
    if not text_col and not description_col:
        raise KeyError('Corpus needs article text or a description/summary/snippet column.')
    title_col = pick_col(raw, ["title", "headline"], required=False)
    date_col = pick_col(raw, ["published_date", "date", "published_at", "created_at", "timestamp"], required=False)
    id_col = pick_col(raw, ["id", "article_id", "story_id", "uuid"], required=False)

    df = pd.DataFrame()
    df['source_row_id'] = range(len(raw))
    df["title"] = raw[title_col].fillna("").map(norm_space) if title_col else ""
    if text_col:
        df['text'] = raw[text_col].fillna('').map(norm_space)
        text_kind = 'article_text'
    else:
        # HackerNoon's published CSV contains headlines and short descriptions.
        # Retain the title as evidence instead of inventing an article body.
        descriptions = raw[description_col].fillna('').map(norm_space)
        df['text'] = [norm_space(f'{title}. {description}') if description else title
                      for title, description in zip(df['title'], descriptions)]
        text_kind = 'title_and_description'

    if date_col:
        df["published_date"] = (
            pd.to_datetime(raw[date_col], errors="coerce", utc=True)
            .dt.strftime("%Y-%m-%d")
            .fillna("")
        )
    else:
        df["published_date"] = ""

    df['source_article_id'] = raw[id_col].fillna('').astype(str) if id_col else ''
    # Source IDs can repeat; content-based IDs remain unique and stable.
    df["article_id"] = [sha1(f"{t}\n{x}")[:20] for t, x in zip(df["title"], df["text"])]

    df = df[df["text"].str.len() >= 80].copy()
    df["dedup_key"] = [
        sha1(norm_space(f"{t}\n{x}").lower())
        for t, x in zip(df["title"], df["text"])
    ]
    before = len(df)
    provenance = df.groupby('dedup_key')['source_row_id'].agg(list).to_dict()
    df['source_row_ids_json'] = df.dedup_key.map(lambda key: json.dumps(provenance[key]))
    priority = set(priority_rows or [])
    df['priority'] = df.dedup_key.map(lambda key: bool(priority & set(provenance[key])))
    df = df.drop_duplicates("dedup_key").drop(columns="dedup_key").reset_index(drop=True)
    print(f"Exact dedup: {before:,} -> {len(df):,}")

    if LAB_MAX_ARTICLES and len(df) > LAB_MAX_ARTICLES:
        keep = df[df.priority]
        if len(keep) > LAB_MAX_ARTICLES:
            raise ValueError('Required sources exceed LAB_MAX_ARTICLES.')
        rest = df[~df.priority].sample(LAB_MAX_ARTICLES - len(keep), random_state=SEED)
        df = pd.concat([keep, rest]).sort_index().reset_index(drop=True)
    pd.DataFrame([{'raw_rows': len(raw), 'nonempty_rows': before, 'deduped_rows': len(provenance),
                   'selected_articles': len(df), 'missing_dates': int(df.published_date.eq('').sum()),
                   'text_kind': text_kind}]).to_csv(
        OUTPUTS / 'preprocessing_stats.csv', index=False)
    return df.drop(columns='priority')

def chunk_text(text, size=220, overlap=40):
    if size <= 0 or overlap < 0 or overlap >= size:
        raise ValueError('Chunk size must be positive and 0 <= overlap < size.')
    words = norm_space(text).split()
    step = max(1, size - overlap)
    out = []
    for start in range(0, len(words), step):
        part = words[start:start+size]
        if not part:
            break
        out.append(" ".join(part))
        if start + size >= len(words):
            break
    return out

def build_chunks(news_df):
    rows = []
    for r in tqdm(news_df.itertuples(index=False), total=len(news_df), desc="Chunking"):
        for i, text in enumerate(chunk_text(r.text, CHUNK_WORDS, CHUNK_OVERLAP_WORDS)):
            rows.append({
                "chunk_id": f"{r.article_id}::c{i:04d}",
                "article_id": r.article_id,
                "source_row_ids_json": getattr(r, 'source_row_ids_json', '[]'),
                "title": r.title,
                "published_date": r.published_date,
                "text": text,
            })
            if LAB_MAX_CHUNKS and len(rows) >= LAB_MAX_CHUNKS:
                return pd.DataFrame(rows)
    return pd.DataFrame(rows)

# raw_df = load_news(DATA_PATH)
# news_df = standardize_news(raw_df)
# chunks_df = build_chunks(news_df)
# display(chunks_df.head())

# 1.5 — Dedup, source identity, coverage-selected sample and chunks
if DATA_PATH.exists():
    try:
        news_df, chunks_df, extraction_source, golden_df, golden_coverage_df = prepare_corpus()
        data_ready = True
        display(chunks_df.head())
        display(golden_coverage_df)
        print('Selected articles/chunks/extraction:', len(news_df), len(chunks_df), len(extraction_source))
    except Exception as error:
        print('PENDING W1/W2:', safe_error(error))
else:
    print('PENDING W1/W2: corpus is missing; no synthetic corpus is substituted.')


Exact dedup: 2,695 -> 2,119


Chunking:   0%|          | 0/1500 [00:00<?, ?it/s]

Chunking: 100%|██████████| 1500/1500 [00:00<00:00, 38576.35it/s]

,chunk_id,article_id,source_row_ids_json,title,published_date,text
0,613de2c24d2456048c08::c0000,613de2c24d2456048c08,[2532],Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google,2023-07-26,Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announ...
1,ae3e878731488df141c0::c0000,ae3e878731488df141c0,"[2537, 3737]",Amazon has drawn thousands to try its AI service competing with Microsoft Google,2023-07-27,Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announced new AI ...
2,0d53dc66fb96e3cb91e2::c0000,0d53dc66fb96e3cb91e2,[2905],Exclusive: Amazon's cloud unit is considering AMD's new AI chips,2023-06-14,Exclusive: Amazon's cloud unit is considering AMD's new AI chips. Amazon Web Services the world''s largest cloud com...
3,8d2f8b148829f4949b97::c0000,8d2f8b148829f4949b97,[3330],More tech companies take White House AI safety pledge,2023-09-12,More tech companies take White House AI safety pledge. IBM Adobe Salesforce and five more tech companies are making ...
4,486309b2b36023d69765::c0000,486309b2b36023d69765,[3357],3 Best Cloud Stocks to Buy in June,2023-06-01,3 Best Cloud Stocks to Buy in June. Microsoft''s market share in cloud computing puts it second with the potential t...


,id,group,corpus_covered,vector_covered,extraction_covered
0,G5000-26,multi-hop,True,True,True
1,G5000-27,cross-doc,True,True,True
2,G5000-28,multi-hop,True,True,True
3,G5000-29,cross-doc,True,True,True
4,G5000-41,factoid,True,True,True


Selected articles/chunks/extraction: 1500 1500 200


### 🎯 AI Coding Agent Challenge A — Near Dedup
Exact hash không bắt được bài repost/near-duplicate.

Hãy dùng AI Agent thiết kế thêm **MinHash/LSH, SimHash hoặc embedding+ANN**.  
**Không chấp nhận** pairwise cosine `O(N²)` trên toàn dataset.

Trong báo cáo nêu:
1. threshold,
2. false positive,
3. cách audit cặp bị merge.

In [6]:
#@title 1.6 — LLM wrapper có retry + JSON parsing
from groq import Groq
groq_client = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

def parse_json_object(text):
    text = str(text).strip()
    text = re.sub(r"^```(?:json)?\s*", "", text, flags=re.I)
    text = re.sub(r"\s*```$", "", text)
    a, b = text.find("{"), text.rfind("}")
    if a < 0 or b <= a:
        raise ValueError("No JSON object found.")
    return json.loads(text[a:b+1])

def groq_chat(messages, model=None, json_mode=False, max_retries=4, purpose='generation'):
    if groq_client is None:
        raise RuntimeError("Thiếu GROQ_API_KEY.")
    model = model or GROQ_MODEL
    if not model:
        raise RuntimeError("Thiếu GROQ_MODEL.")

    last = None
    for attempt in range(max_retries):
        try:
            kwargs = {
                "model": model,
                "messages": messages,
                "temperature": 0.0,
                'timeout': 60.0,
            }
            if json_mode:
                kwargs["response_format"] = {"type": "json_object"}
            if model.startswith('openai/gpt-oss-'):
                kwargs['reasoning_effort'] = GROQ_REASONING_EFFORT

            resp = groq_client.chat.completions.create(**kwargs)
            usage = {}
            if getattr(resp, "usage", None):
                usage = {
                    "prompt_tokens": getattr(resp.usage, "prompt_tokens", None),
                    "completion_tokens": getattr(resp.usage, "completion_tokens", None),
                    "total_tokens": getattr(resp.usage, "total_tokens", None),
                }
            USAGE_LOG.append({'provider': 'groq', 'model': model, 'purpose': purpose, **usage})
            return resp.choices[0].message.content, usage
        except Exception as e:
            last = e
            if attempt == max_retries - 1 or getattr(e, 'status_code', None) in (400, 401, 403, 404):
                break
            time.sleep(min(20, 2**attempt + random.random()))
    raise RuntimeError(safe_error(last))

def groq_json(system, user, model=None, purpose='structured'):
    text, usage = groq_chat(
        [{"role": "system", "content": system},
         {"role": "user", "content": user}],
        model=model,
        json_mode=True,
        purpose=purpose,
    )
    return parse_json_object(text), usage

# 1.6 — Retry/JSON implementation and provider configuration
print('Generator model:', GROQ_MODEL, '| Judge:', JUDGE_PROVIDER, JUDGE_MODEL)
print('Credentials are read from .env/Colab Secrets and never printed.')
if llm_ready:
    probe, _ = groq_json('Return strict JSON.', 'Return {"ok": true}', purpose='preflight')
    assert probe.get('ok') is True
    print('Groq JSON request passed.')


Generator model: openai/gpt-oss-120b | Judge: groq openai/gpt-oss-120b
Credentials are read from .env/Colab Secrets and never printed.


Groq JSON request passed.


## 1.7 — Coreference Resolution

Yêu cầu:
- chỉ resolve đại từ khi antecedent rõ trong cùng chunk,
- không invent fact,
- giữ nguyên số/ngày/ticker/product,
- ambiguity → giữ nguyên và log `unresolved_mentions`.

**Failure mode quan trọng:** false coreference → false edge.

In [7]:
#@title 1.7 — Coreference resolution theo batch
COREF_SYSTEM = """
You are a conservative coreference-resolution component for a knowledge-graph pipeline.
Resolve pronouns and generic references only when the antecedent is clearly supported in the same chunk.
Never invent facts. Preserve dates, numbers, tickers and product names.
Return strict JSON only.
""".strip()

def resolve_coref_batch(batch_df):
    payload = [{"chunk_id": r.chunk_id, "text": r.text}
               for r in batch_df.itertuples(index=False)]

    prompt = f"""
Resolve coreferences.

Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "resolved_text": "...",
      "unresolved_mentions": ["..."]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()

    cached = cached_json('coref', {'version': PROMPT_VERSION, 'model': GROQ_MODEL, 'reasoning_effort': GROQ_REASONING_EFFORT,
                                 'system': COREF_SYSTEM, 'prompt': prompt},
                         lambda: dict(zip(('result', 'usage'), groq_json(COREF_SYSTEM, prompt, purpose='coreference'))))
    obj, usage = cached['result'], cached['usage']
    if not isinstance(obj.get('items'), list):
        raise ValueError('Coreference JSON items must be a list.')
    by_id = {x.get("chunk_id"): x for x in obj['items'] if isinstance(x, dict)}

    rows = []
    for r in batch_df.itertuples(index=False):
        item = by_id.get(r.chunk_id, {})
        resolved = norm_space(item.get('resolved_text') or r.text)
        unresolved = item.get('unresolved_mentions', [])
        status = 'resolved' if resolved != r.text else 'unchanged'
        if not item or not isinstance(unresolved, list):
            resolved, unresolved, status = r.text, ['INVALID_OR_MISSING_COREF_ITEM'], 'rejected'
        if Counter(re.findall(r'\b\d+(?:[.,]\d+)*\b', r.text)) != Counter(re.findall(r'\b\d+(?:[.,]\d+)*\b', resolved)):
            resolved, unresolved, status = r.text, ['COREF_CHANGED_NUMBERS'], 'rejected'
        rows.append({
            "chunk_id": r.chunk_id,
            "resolved_text": resolved,
            "unresolved_mentions": unresolved,
            'coref_status': status,
        })
    return pd.DataFrame(rows), usage

def run_coref(chunks_subset, batch_size=None):
    batch_size = COREF_BATCH_SIZE if batch_size is None else batch_size
    out = []
    for start in tqdm(range(0, len(chunks_subset), batch_size), desc="Coref"):
        batch = chunks_subset.iloc[start:start+batch_size]
        try:
            df, _ = resolve_coref_batch(batch)
        except Exception as error:
            df = pd.DataFrame({
                "chunk_id": batch["chunk_id"].tolist(),
                "resolved_text": batch["text"].tolist(),
                "unresolved_mentions": [["COREF_BATCH_FAILED"] for _ in range(len(batch))],
                'coref_status': ['failed'] * len(batch),
            })
            print('Coreference batch failed:', safe_error(error))
        out.append(df)
    result = pd.concat(out, ignore_index=True) if out else pd.DataFrame()
    result.to_csv(OUTPUTS / 'coref_checkpoint.csv', index=False)
    if not result.empty:
        audit = result.copy()
        original = chunks_subset.set_index('chunk_id')['text'].to_dict()
        audit['original_text'] = audit.chunk_id.map(original)
        audit['unresolved_mentions'] = audit.unresolved_mentions.map(json.dumps)
        audit.to_csv(OUTPUTS / 'coref_audit.csv', index=False)
    return result

# extraction_source = chunks_df.head(EXTRACTION_MAX_CHUNKS).copy()
# coref_df = run_coref(extraction_source)
# extraction_source = extraction_source.merge(coref_df, on="chunk_id", how="left")

# 1.7 — Conservative coreference with per-batch checkpoints and numeric guard
if data_ready and llm_ready:
    coref_df = run_coref(extraction_source)
    extraction_source = extraction_source.merge(coref_df, on='chunk_id', validate='one_to_one')
    display(coref_df.head())
    display(coref_df.coref_status.value_counts())
else:
    print('PENDING W2: coreference requires the corpus and configured LLM.')


Coref:   0%|          | 0/25 [00:00<?, ?it/s]

Coref:  52%|█████▏    | 13/25 [00:00<00:00, 123.76it/s]

Coref: 100%|██████████| 25/25 [00:00<00:00, 142.50it/s]

,chunk_id,resolved_text,unresolved_mentions,coref_status
0,613de2c24d2456048c08::c0000,Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announ...,[],unchanged
1,ae3e878731488df141c0::c0000,Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announced new AI ...,[],unchanged
2,0d53dc66fb96e3cb91e2::c0000,Exclusive: Amazon's cloud unit is considering AMD's new AI chips. Amazon Web Services the world''s largest cloud com...,[],unchanged
3,8d2f8b148829f4949b97::c0000,More tech companies take White House AI safety pledge. IBM Adobe Salesforce and five more tech companies are making ...,[],unchanged
4,486309b2b36023d69765::c0000,3 Best Cloud Stocks to Buy in June. Microsoft''s market share in cloud computing puts it second with the potential t...,[],unchanged


coref_status
unchanged    178
resolved      21
rejected       1
Name: count, dtype: int64

# PHẦN 2 — TRIPLE EXTRACTION & NEO4J BULK INSERT (15–45')

## Graph schema
**Nodes:** `Company`, `Person`, `Technology` + base label `Entity`.

**Relations:** `ACQUIRED`, `DEVELOPED`, `INVESTED_IN`, `FOUNDED`, `WORKED_AT`, `PARTNERED_WITH`, `USES`, `LEADS`.

**Mỗi edge bắt buộc:** `source_chunk_id`, `published_date`; khuyến nghị thêm `evidence`, `confidence`.

> Relation type phải qua allowlist trước khi ghép vào Cypher.

In [8]:
#@title 2.1 — NER + RE extraction
ALLOWED_NODE_TYPES = {"Company", "Person", "Technology"}
ALLOWED_RELATIONS = {
    "ACQUIRED", "DEVELOPED", "INVESTED_IN", "FOUNDED",
    "WORKED_AT", "PARTNERED_WITH", "USES", "LEADS"
}

EXTRACT_SYSTEM = f"""
Extract a high-precision knowledge graph from tech-news text.
Allowed node types: {sorted(ALLOWED_NODE_TYPES)}
Allowed relations: {sorted(ALLOWED_RELATIONS)}
Use only explicitly supported facts. Prefer precision over recall.
Every relation needs short evidence copied verbatim from original_text, not rewritten text.
Do not turn a plan, consideration or speculation into a confirmed relation. Return strict JSON only.
""".strip()

def extract_batch(batch_df):
    payload = [{
        "chunk_id": r.chunk_id,
        "published_date": r.published_date,
        "text": getattr(r, "resolved_text", None) or r.text,
        'original_text': r.text,
    } for r in batch_df.itertuples(index=False)]

    prompt = f"""
Return:
{{
  "items": [
    {{
      "chunk_id": "...",
      "relations": [
        {{
          "source": "...",
          "source_type": "Company|Person|Technology",
          "relation": "ALLOWED_RELATION",
          "target": "...",
          "target_type": "Company|Person|Technology",
          "evidence": "...",
          "confidence": 0.0
        }}
      ]
    }}
  ]
}}

INPUT:
{json.dumps(payload, ensure_ascii=False)}
""".strip()
    cached = cached_json('extract', {'version': PROMPT_VERSION, 'model': GROQ_MODEL, 'reasoning_effort': GROQ_REASONING_EFFORT,
                                   'system': EXTRACT_SYSTEM, 'prompt': prompt},
                         lambda: dict(zip(('result', 'usage'), groq_json(EXTRACT_SYSTEM, prompt, purpose='relation_extraction'))))
    return cached['result'], cached['usage']

def run_extraction(source_df, batch_size=None):
    batch_size = EXTRACTION_BATCH_SIZE if batch_size is None else batch_size
    meta = source_df.set_index("chunk_id")["published_date"].to_dict()
    original = source_df.set_index('chunk_id')['text'].map(norm_space).to_dict()
    triples, errors = [], []

    for start in tqdm(range(0, len(source_df), batch_size), desc="NER+RE"):
        batch = source_df.iloc[start:start+batch_size]
        try:
            obj, _ = extract_batch(batch)
        except Exception as e:
            errors.append({"start": start, "error": safe_error(e)})
            continue

        if not isinstance(obj.get('items'), list):
            errors.append({'start': start, 'error': 'Invalid JSON items list'})
            continue
        for item in obj['items']:
            if not isinstance(item, dict):
                errors.append({'start': start, 'error': 'Invalid JSON item'})
                continue
            cid = item.get("chunk_id")
            if cid not in set(batch.chunk_id):
                errors.append({'start': start, 'error': 'Unknown batch chunk_id'})
                continue
            relations = item.get('relations', [])
            if not isinstance(relations, list):
                errors.append({'start': start, 'error': 'Invalid relations list'})
                continue
            for x in relations:
                if not isinstance(x, dict):
                    errors.append({'start': start, 'error': 'Invalid relation object'})
                    continue
                s, t = norm_space(x.get("source")), norm_space(x.get("target"))
                st, tt, rel = x.get("source_type"), x.get("target_type"), x.get("relation")
                if not s or not t:
                    continue
                if st not in ALLOWED_NODE_TYPES or tt not in ALLOWED_NODE_TYPES:
                    continue
                if rel not in ALLOWED_RELATIONS:
                    errors.append({'start': start, 'error': 'Relation outside allowlist'})
                    continue
                evidence = norm_space(x.get('evidence'))
                try:
                    confidence = float(x.get('confidence'))
                except (TypeError, ValueError):
                    confidence = math.nan
                date = norm_space(meta[cid])
                if (not evidence or evidence not in original[cid] or
                        not math.isfinite(confidence) or not 0 <= confidence <= 1 or
                        pd.isna(pd.to_datetime(date, utc=True, errors='coerce'))):
                    errors.append({'start': start, 'chunk_id': cid,
                                   'error': 'Rejected invalid evidence/confidence/date'})
                    continue
                triples.append({
                    "source_raw": s,
                    "source_type": st,
                    "relation": rel,
                    "target_raw": t,
                    "target_type": tt,
                    "source_chunk_id": cid,
                    "published_date": meta[cid] or "",
                    "evidence": evidence,
                    "confidence": confidence,
                })

    columns = ['source_raw', 'source_type', 'relation', 'target_raw', 'target_type',
               'source_chunk_id', 'published_date', 'evidence', 'confidence']
    result = pd.DataFrame(triples, columns=columns).drop_duplicates(
        ['source_raw', 'source_type', 'relation', 'target_raw', 'target_type', 'source_chunk_id'])
    error_df = pd.DataFrame(errors, columns=['start', 'chunk_id', 'error'])
    result.to_csv(OUTPUTS / 'triples_checkpoint.csv', index=False)
    error_df.to_csv(OUTPUTS / 'extraction_errors.csv', index=False)
    return result, error_df

# raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
# display(raw_triples_df.head())

# 2.1 — Schema-checked triples; evidence must be copied from original text
if data_ready and llm_ready:
    raw_triples_df, extraction_errors_df = run_extraction(extraction_source)
    display(raw_triples_df.head())
    display(extraction_errors_df.head())
    print('Valid triples:', len(raw_triples_df), '| errors/rejections:', len(extraction_errors_df))
else:
    raw_triples_df = pd.DataFrame()
    print('PENDING W3: NER/RE has not run.')


NER+RE:   0%|          | 0/25 [00:00<?, ?it/s]

NER+RE:  64%|██████▍   | 16/25 [00:00<00:00, 145.91it/s]

NER+RE: 100%|██████████| 25/25 [00:00<00:00, 156.05it/s]

,source_raw,source_type,relation,target_raw,target_type,source_chunk_id,published_date,evidence,confidence
0,Amazon,Company,USES,Cohere,Company,613de2c24d2456048c08::c0000,2023-07-26,technology access from the startup Cohere,1.0
1,Hewlett Packard Enterprise,Company,ACQUIRED,Axis Security,Company,5107f0cd19116339b25d::c0000,2023-03-02,entered into a definitive agreement to acquire Axis Security,1.0
2,Google Cloud,Company,USES,Llama 2,Technology,7ca725290aab95bd56a7::c0000,2023-08-29,announcing availability of Llama 2,1.0
3,Google Cloud,Company,USES,Falcon LLM,Technology,7ca725290aab95bd56a7::c0000,2023-08-29,Technology Innovative Institute''s Falcon LLM a popular open-source model,1.0
4,Google Cloud,Company,USES,Claude 2,Technology,7ca725290aab95bd56a7::c0000,2023-08-29,pre-announcing Claude 2 from Anthropic,1.0


,start,chunk_id,error
0,0,7ca725290aab95bd56a7::c0000,Rejected invalid evidence/confidence/date
1,24,5de01398a4298b732892::c0000,Rejected invalid evidence/confidence/date
2,24,5de01398a4298b732892::c0000,Rejected invalid evidence/confidence/date
3,24,a35c246dcfe101d64398::c0000,Rejected invalid evidence/confidence/date
4,48,3a76b31b81939716924c::c0000,Rejected invalid evidence/confidence/date


Valid triples: 52 | errors/rejections: 10


## 2.2 — Entity Resolution bằng Vector Similarity

Pipeline:
1. Manual aliases cho ticker/tên rất phổ biến.
2. Embedding ANN candidate.
3. Lexical guard để giảm false merge.
4. Xuất audit table.

### 🎯 AI Coding Agent Challenge B
Cải tiến guard cho:
- ticker,
- suffix `Inc./Corp./Ltd.`,
- product chứa company name,
- người trùng họ/tên gần giống.

In [9]:
#@title 2.2 — Entity resolution
CORP_SUFFIXES = {"inc","incorporated","corp","corporation","ltd","limited","llc","plc","co","company"}
MANUAL_ALIASES = {
    "msft": "Microsoft",
    "microsoft corp": "Microsoft",
    "microsoft corporation": "Microsoft",
    "goog": "Google",
    "googl": "Google",
    "google llc": "Google",
    "meta platforms": "Meta",
    "meta platforms inc": "Meta",
    "aapl": "Apple",
    "apple inc": "Apple",
}

def norm_entity(name):
    s = unicodedata.normalize("NFKC", norm_space(name)).lower()
    s = re.sub(r"[^\w\s\-\.]", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def strip_suffix(name):
    toks = norm_entity(name).replace(".", "").split()
    while toks and toks[-1] in CORP_SUFFIXES:
        toks.pop()
    return " ".join(toks)

def merge_guard(a, b, typ=None):
    na, nb = strip_suffix(a), strip_suffix(b)
    if na == nb:
        return True
    tokens_a, tokens_b = na.split(), nb.split()
    if not tokens_a or not tokens_b:
        return False
    if typ == 'Person' and tokens_a[0] != tokens_b[0]:
        return False
    # A company's bare name and its longer product name are distinct entities.
    if set(tokens_a) < set(tokens_b) or set(tokens_b) < set(tokens_a):
        return False
    return SequenceMatcher(None, na, nb).ratio() >= 0.72

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
embedder = None

def get_embedder():
    global embedder
    if embedder is None:
        embedder = SentenceTransformer(EMBED_MODEL)
    return embedder

class UF:
    def __init__(self, n):
        self.p = list(range(n))
    def find(self, x):
        if self.p[x] != x:
            self.p[x] = self.find(self.p[x])
        return self.p[x]
    def union(self, a, b):
        a, b = self.find(a), self.find(b)
        if a != b:
            self.p[b] = a

def build_resolution_map(raw_triples_df, threshold=0.90, top_k=5):
    mentions = []
    for r in raw_triples_df.itertuples(index=False):
        mentions += [(r.source_type, r.source_raw), (r.target_type, r.target_raw)]

    counts = Counter((t, norm_entity(n)) for t, n in mentions)
    display_name = {}
    for t, n in mentions:
        display_name.setdefault((t, norm_entity(n)), n)

    mapping, audit = {}, []

    for key in counts:
        t, norm = key
        if t == 'Company' and norm in MANUAL_ALIASES:
            mapping[key] = MANUAL_ALIASES[norm]
            audit.append({
                "type": t, "left": display_name[key],
                "right": MANUAL_ALIASES[norm],
                "similarity": 1.0, "decision": "MERGE_MANUAL"
            })

    for typ in sorted(ALLOWED_NODE_TYPES):
        keys = [k for k in counts if k[0] == typ and k not in mapping]
        if not keys:
            continue
        names = [display_name[k] for k in keys]
        vecs = get_embedder().encode(
            names, batch_size=128, show_progress_bar=False,
            normalize_embeddings=True
        ).astype("float32")

        index = faiss.IndexHNSWFlat(vecs.shape[1], 32, faiss.METRIC_INNER_PRODUCT)
        index.hnsw.efConstruction = 40
        index.hnsw.efSearch = 64
        index.add(vecs)
        sims, nbrs = index.search(vecs, min(top_k, len(names)))
        uf = UF(len(names))

        seen_pairs = set()
        for i in range(len(names)):
            for score, j in zip(sims[i], nbrs[i]):
                pair = tuple(sorted((i, int(j))))
                if j < 0 or i == j or pair in seen_pairs:
                    continue
                seen_pairs.add(pair)
                guard_ok = merge_guard(names[i], names[j], typ=typ)
                ok = float(score) >= threshold and guard_ok
                audit.append({
                    "type": typ, "left": names[i], "right": names[j],
                    "similarity": float(score),
                    "decision": 'MERGE_VECTOR' if ok else ('REJECT_THRESHOLD' if float(score) < threshold else 'REJECT_GUARD'),
                    'reason': 'similarity_below_threshold' if float(score) < threshold else
                              ('guard_passed' if guard_ok else 'name_or_product_guard'),
                })
                if ok:
                    uf.union(i, j)

        groups = defaultdict(list)
        for i in range(len(names)):
            groups[uf.find(i)].append(i)

        for idxs in groups.values():
            best = sorted(
                idxs,
                key=lambda i: (-counts[keys[i]], len(names[i]), names[i].lower())
            )[0]
            canonical = names[best]
            for i in idxs:
                mapping[keys[i]] = canonical

    for key in counts:
        mapping.setdefault(key, display_name[key])

    audit_df = pd.DataFrame(audit, columns=['type', 'left', 'right', 'similarity', 'decision', 'reason'])
    for side in ['left', 'right']:
        audit_df['canonical_' + side] = [mapping.get((row.type, norm_entity(getattr(row, side))), getattr(row, side))
                                        for row in audit_df.itertuples(index=False)]
    audit_df.to_csv(OUTPUTS / 'entity_resolution_audit.csv', index=False)
    return mapping, audit_df

def canonicalize_triples(raw_df, mapping):
    df = raw_df.copy()
    def canon(name, typ):
        n = norm_entity(name)
        return mapping.get((typ, n), MANUAL_ALIASES.get(n, name) if typ == 'Company' else name)

    df["source_name"] = [canon(n,t) for n,t in zip(df.source_raw, df.source_type)]
    df["target_name"] = [canon(n,t) for n,t in zip(df.target_raw, df.target_type)]
    df["source_name_norm"] = df.source_name.map(norm_entity)
    df["target_name_norm"] = df.target_name.map(norm_entity)
    df["source_id"] = [sha1(f"{LAB_NAMESPACE}:{t}:{n}")[:24] for t,n in zip(df.source_type, df.source_name_norm)]
    df["target_id"] = [sha1(f"{LAB_NAMESPACE}:{t}:{n}")[:24] for t,n in zip(df.target_type, df.target_name_norm)]
    return df[df.source_id != df.target_id].reset_index(drop=True)

# entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df)
# triples_df = canonicalize_triples(raw_triples_df, entity_map)
# display(entity_resolution_audit_df.head(20))

# 2.2 — Alias → normalized HNSW candidates → lexical/type guard → Union-Find
if not raw_triples_df.empty:
    entity_map, entity_resolution_audit_df = build_resolution_map(raw_triples_df, threshold=0.90)
    triples_df = canonicalize_triples(raw_triples_df, entity_map)
    display(entity_resolution_audit_df.head(20))
    print('Audit rows:', len(entity_resolution_audit_df))
else:
    print('PENDING W3: no extracted triples to resolve.')


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1370.81it/s]

,type,left,right,similarity,decision,reason,canonical_left,canonical_right
0,Company,Amazon,Amazon Web Services,0.646989,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Web Services
1,Company,Amazon,Amazon Publisher Services,0.585162,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Publisher Services
2,Company,Amazon,Salesforce,0.537856,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Salesforce
3,Company,Amazon,Apple,0.485086,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Apple
4,Company,Cohere,Linde,0.367736,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Linde
5,Company,Cohere,Worldpay,0.351073,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Worldpay
6,Company,Cohere,Odyssey,0.338944,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Odyssey
7,Company,Cohere,Thales,0.336762,REJECT_THRESHOLD,similarity_below_threshold,Cohere,Thales
8,Company,Hewlett Packard Enterprise,Microsoft,0.488284,REJECT_THRESHOLD,similarity_below_threshold,Hewlett Packard Enterprise,Microsoft
9,Company,Hewlett Packard Enterprise,Airbus,0.405509,REJECT_THRESHOLD,similarity_below_threshold,Hewlett Packard Enterprise,Airbus


Audit rows: 282


In [10]:
#@title 2.3 — Node table + UNWIND bulk insert
def build_nodes(triples_df):
    rows = []
    for r in triples_df.itertuples(index=False):
        rows += [
            {"id":r.source_id,"name":r.source_name,"name_norm":r.source_name_norm,"type":r.source_type,"alias":r.source_raw},
            {"id":r.target_id,"name":r.target_name,"name_norm":r.target_name_norm,"type":r.target_type,"alias":r.target_raw},
        ]
    tmp = pd.DataFrame(rows)
    if tmp.empty:
        return tmp

    out = []
    for (node_id,name,name_norm,typ), g in tmp.groupby(["id","name","name_norm","type"]):
        aliases = sorted(set(g["alias"].map(norm_space)))
        out.append({
            "id":node_id, "name":name, "name_norm":name_norm, "type":typ,
            "aliases":aliases,
            "aliases_norm":sorted(set(norm_entity(x) for x in aliases))
        })
    return pd.DataFrame(out)

def batches(records, size=1000):
    for i in range(0, len(records), size):
        yield records[i:i+size]

def bulk_insert_nodes(nodes_df, batch_size=1000):
    if not set(nodes_df.type) <= ALLOWED_NODE_TYPES:
        raise ValueError('Node type outside allowlist.')
    for typ in sorted(ALLOWED_NODE_TYPES):
        part = nodes_df[nodes_df.type == typ]
        if part.empty:
            continue
        query = f"""
        UNWIND $rows AS row
        MERGE (n:Entity {{id: row.id}})
        SET n:{typ}:Lab19,
            n.lab_namespace=$namespace,
            n.name=row.name,
            n.name_norm=row.name_norm,
            n.entity_type=row.type,
            n.aliases=row.aliases,
            n.aliases_norm=row.aliases_norm
        """
        for b in batches(part.to_dict("records"), batch_size):
            run_cypher(query, rows=b)

def bulk_insert_edges(triples_df, batch_size=1000):
    required = {"source_chunk_id","published_date"}
    if not required.issubset(triples_df.columns):
        raise ValueError("Missing edge provenance.")
    for field in required | {'evidence'}:
        if triples_df[field].fillna('').astype(str).str.strip().eq('').any():
            raise ValueError(f'Empty edge provenance: {field}')
    if not set(triples_df.relation) <= ALLOWED_RELATIONS:
        raise ValueError('Relationship outside allowlist.')
    confidence = pd.to_numeric(triples_df.confidence, errors='coerce')
    if confidence.isna().any() or not confidence.between(0, 1).all():
        raise ValueError('Invalid confidence.')
    if pd.to_datetime(triples_df.published_date, utc=True, errors='coerce').isna().any():
        raise ValueError('Invalid published_date.')

    for rel in sorted(ALLOWED_RELATIONS):
        part = triples_df[triples_df.relation == rel]
        if part.empty:
            continue

        query = f"""
        UNWIND $rows AS row
        MATCH (s:Lab19 {{id: row.source_id, lab_namespace:$namespace}})
        MATCH (t:Lab19 {{id: row.target_id, lab_namespace:$namespace}})
        MERGE (s)-[r:{rel} {{source_chunk_id: row.source_chunk_id}}]->(t)
        SET r.lab_namespace=$namespace,
            r.published_date=row.published_date,
            r.evidence=row.evidence,
            r.confidence=row.confidence
        """

        cols = ["source_id","target_id","source_chunk_id","published_date","evidence","confidence"]
        for b in batches(part[cols].to_dict("records"), batch_size):
            run_cypher(query, rows=b)

# nodes_df = build_nodes(triples_df)
# bulk_insert_nodes(nodes_df)
# bulk_insert_edges(triples_df)

# 2.3 — Bulk UNWIND; nodes and relationships are scoped to LAB_NAMESPACE
if not raw_triples_df.empty and neo4j_ready:
    nodes_df = build_nodes(triples_df)
    bulk_insert_nodes(nodes_df)
    bulk_insert_edges(triples_df)
    graph_ready = True
    print('Ingested nodes/triples:', len(nodes_df), len(triples_df))
else:
    print('PENDING W3: ingestion has not run.')


Ingested nodes/triples: 97 52


In [11]:
#@title 2.4 — Sanity checks
def graph_checks():
    invalid = run_cypher("""
    MATCH (:Lab19 {lab_namespace:$namespace})-[r]->(:Lab19 {lab_namespace:$namespace})
    WHERE r.lab_namespace=$namespace AND
      (trim(coalesce(r.source_chunk_id,''))='' OR trim(coalesce(r.published_date,''))='' OR
       trim(coalesce(r.evidence,''))='')
    RETURN count(r) AS n
    """)[0]["n"]

    counts = {
        "nodes": run_cypher("MATCH (n:Lab19 {lab_namespace:$namespace}) RETURN count(n) AS n")[0]["n"],
        "edges": run_cypher("MATCH ()-[r]->() WHERE r.lab_namespace=$namespace RETURN count(r) AS n")[0]["n"],
        "invalid_provenance_edges": invalid,
    }
    print(counts)
    assert invalid == 0

    top = pd.DataFrame(run_cypher("""
    MATCH (n:Lab19 {lab_namespace:$namespace})
    OPTIONAL MATCH (n)-[r]-()
    WHERE r.lab_namespace=$namespace
    WITH n, count(r) AS degree
    RETURN n.id AS id, n.name AS name, n.entity_type AS type, degree
    ORDER BY degree DESC LIMIT 15
    """))
    display(top)
    pd.DataFrame([counts]).to_csv(OUTPUTS / 'graph_sanity.csv', index=False)
    top.to_csv(OUTPUTS / 'top_degree_entities.csv', index=False)
    return counts, top

# graph_counts, top_degree_df = graph_checks()

# 2.4 — Includes NULL, empty-string provenance and a real top-degree table
if graph_ready:
    graph_counts, top_degree_df = graph_checks()
else:
    print('PENDING W3: graph integrity has not been measured.')


{'nodes': 97, 'edges': 52, 'invalid_provenance_edges': 0}


,id,name,type,degree
0,18a2f0e496fb60f2b80e8775,Google Cloud,Company,3
1,fc94d4789fde1f6e802475fd,L&T Technology Services,Company,2
2,51e652cc3cb71d3060de600c,Thales,Company,2
3,2618ac416fd176fc8751f56d,Data Innovations,Company,2
4,da83c25f2271b3c0df0cad56,AGC Biologics,Company,2
5,5d63f575c2fcc0b8f6e2e3c8,Freshworks,Company,2
6,1b8f7284287734ef1f96795d,Sineng Electric,Company,1
7,2a53b5e8507255a685aa84d7,Aqara,Company,1
8,0fccf849f555ed7fc05dba5e,Qualcomm,Company,1
9,24a29c9a8a01d23c25f3bb61,Associated Press,Company,1


# PHẦN 3 — FLAT RAG & HYBRID GRAPHRAG (45–75')

## Flat RAG baseline
Dùng cùng embedding/generator để comparison tập trung vào retrieval architecture.

In [12]:
#@title 3.1 — Flat RAG
flat_index = None
flat_store = None
entity_match_vectors = None
entity_match_store = None

def build_flat_index(chunks_df):
    global flat_index, flat_store
    if chunks_df.empty:
        raise ValueError('Cannot index an empty corpus.')
    vecs = get_embedder().encode(
        chunks_df.text.fillna("").tolist(),
        batch_size=128, show_progress_bar=True,
        normalize_embeddings=True
    ).astype("float32")

    flat_index = faiss.IndexFlatIP(vecs.shape[1])
    flat_index.add(vecs)
    flat_store = chunks_df.reset_index(drop=True).copy()
    print("Flat vectors:", flat_index.ntotal)

def retrieve_flat_context(query, k=6):
    if flat_index is None or flat_store is None:
        raise RuntimeError('Build the Flat RAG index before retrieval.')
    qv = get_embedder().encode(
        [query], normalize_embeddings=True, show_progress_bar=False
    ).astype("float32")
    scores, ids = flat_index.search(qv, min(k, flat_index.ntotal))

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx < 0:
            continue
        r = flat_store.iloc[int(idx)]
        rows.append({
            "score":float(score), "chunk_id":r.chunk_id,
            "published_date":r.published_date, "text":r.text
        })

    df = pd.DataFrame(rows)
    context = "\n\n".join(
        f"[chunk_id={r.chunk_id} | date={r.published_date} | score={r.score:.3f}]\n{r.text}"
        for r in df.itertuples(index=False)
    )
    return context, df

# build_flat_index(chunks_df)

# 3.1 — Flat RAG index over the same corpus chunks
if data_ready:
    build_flat_index(chunks_df)
    print('Flat baseline ready; top-k = 6.')
else:
    print('PENDING W4: Flat index requires corpus chunks.')


Batches:   0%|          | 0/12 [00:00<?, ?it/s]

Batches:   8%|▊         | 1/12 [00:15<02:49, 15.44s/it]

Batches:  17%|█▋        | 2/12 [00:22<01:43, 10.39s/it]

Batches:  25%|██▌       | 3/12 [00:28<01:17,  8.63s/it]

Batches:  33%|███▎      | 4/12 [00:38<01:12,  9.12s/it]

...

Batches:  83%|████████▎ | 10/12 [01:20<00:14,  7.43s/it]

Batches:  92%|█████████▏| 11/12 [01:27<00:07,  7.12s/it]

Batches: 100%|██████████| 12/12 [01:30<00:00,  6.06s/it]

Batches: 100%|██████████| 12/12 [01:30<00:00,  7.56s/it]

Flat vectors: 1500
Flat baseline ready; top-k = 6.


## Graph retrieval flow
1. LLM trích seed entities.
2. Match seed trong Neo4j; fuzzy fallback bằng embedding.
3. BFS tối đa `max_hops`.
4. Nếu node degree > 100 → chỉ lấy tối đa 50 edge mới nhất.
5. Global edge cap để tránh context explosion.
6. Textualize subgraph có provenance.

In [13]:
#@title 3.2 — Seed matching
SEED_SYSTEM = """
Extract useful seed entities for graph retrieval.
Allowed types: Company, Person, Technology.
Do not answer the question. Return strict JSON only.
""".strip()

def extract_seeds(query):
    obj, _ = groq_json(SEED_SYSTEM, f"""
Question: {query}
Return {{"seeds":[{{"name":"...","type":"Company|Person|Technology|null"}}]}}
""", purpose='seed_extraction')
    return [
        {"name":norm_space(x.get("name")),
         "type":x.get("type") if x.get("type") in ALLOWED_NODE_TYPES else None}
        for x in obj.get("seeds", [])
        if norm_space(x.get("name"))
    ]

def build_entity_matcher(nodes_df):
    global entity_match_vectors, entity_match_store
    entity_match_store = nodes_df.reset_index(drop=True).copy()
    entity_match_vectors = get_embedder().encode(
        entity_match_store.name.tolist(),
        batch_size=128, show_progress_bar=False,
        normalize_embeddings=True
    ).astype("float32")

def match_seeds(query, fuzzy_threshold=0.66):
    matched = []
    for seed in extract_seeds(query):
        exact = run_cypher("""
        MATCH (n:Lab19 {lab_namespace:$namespace})
        WHERE (n.name_norm=$name OR $name IN coalesce(n.aliases_norm,[]))
          AND ($typ IS NULL OR n.entity_type=$typ)
        RETURN n.id AS id, n.name AS name, n.entity_type AS type
        LIMIT 5
        """, name=norm_entity(seed["name"]), typ=seed["type"])

        if exact:
            matched += exact
            continue

        if entity_match_vectors is None:
            continue

        mask = np.ones(len(entity_match_store), dtype=bool)
        if seed["type"]:
            mask = entity_match_store.type.eq(seed["type"]).to_numpy()
        idxs = np.flatnonzero(mask)
        if not len(idxs):
            continue

        qv = get_embedder().encode(
            [seed["name"]], normalize_embeddings=True, show_progress_bar=False
        ).astype("float32")[0]
        sims = entity_match_vectors[idxs] @ qv
        j = int(np.argmax(sims))
        if float(sims[j]) >= fuzzy_threshold:
            r = entity_match_store.iloc[int(idxs[j])]
            matched.append({"id":r.id,"name":r.name,"type":r.type})

    return list({x["id"]: x for x in matched}.values())

# build_entity_matcher(nodes_df)

# 3.2 — Seed resolution: exact aliases then embedding fallback 0.66
if graph_ready:
    build_entity_matcher(nodes_df)
    retrieval_ready = True
else:
    print('PENDING W4: entity matcher requires canonical graph nodes.')


In [14]:
#@title 3.3 — Graph traversal + super-node mitigation
SUPER_NODE_DEGREE = 100
SUPER_NODE_EDGE_CAP = 50
GLOBAL_EDGE_CAP = 250
MAX_GRAPH_CONTEXT_CHARS = 14000

def node_degree(node_id):
    return int(run_cypher("""
    MATCH (n:Lab19 {id:$id, lab_namespace:$namespace})
    OPTIONAL MATCH (n)-[r]-()
    WHERE r.lab_namespace=$namespace
    RETURN count(r) AS degree
    """, id=node_id)[0]["degree"])

def recent_edges(node_id, limit):
    return run_cypher("""
    MATCH (n:Lab19 {id:$id, lab_namespace:$namespace})
    MATCH (n)-[r]-(m:Lab19 {lab_namespace:$namespace})
    WHERE r.lab_namespace=$namespace
    RETURN
      startNode(r).id AS source_id,
      startNode(r).name AS source_name,
      startNode(r).entity_type AS source_type,
      type(r) AS relation,
      endNode(r).id AS target_id,
      endNode(r).name AS target_name,
      endNode(r).entity_type AS target_type,
      r.source_chunk_id AS source_chunk_id,
      r.published_date AS published_date,
      r.evidence AS evidence,
      m.id AS neighbor_id
    ORDER BY coalesce(r.published_date,'') DESC
    LIMIT $limit
    """, id=node_id, limit=int(limit))

def textualize(edges):
    edges = sorted(edges, key=lambda e:e.get("published_date") or "", reverse=True)
    lines, used = [], 0
    for e in edges:
        line = (
            f"{e['source_name']} [{e['source_type']}] -{e['relation']}-> "
            f"{e['target_name']} [{e['target_type']}] "
            f"| date={e.get('published_date') or 'unknown'} "
            f"| chunk={e.get('source_chunk_id') or 'unknown'}"
        )
        if e.get("evidence"):
            line += f" | evidence={norm_space(e['evidence'])}"
        if used + len(line) + 1 > MAX_GRAPH_CONTEXT_CHARS:
            break
        lines.append(line)
        used += len(line) + 1
    return "\n".join(lines)

def retrieve_graph_context(query, max_hops=2, edge_limit=50, return_debug=False):
    seeds = match_seeds(query)
    if not seeds:
        out = {"context":"","edges":pd.DataFrame(),
               "diagnostics":{"reason":"NO_SEED","supernode_events":[]}}
        return out if return_debug else ""

    frontier = deque((x["id"],0) for x in seeds)
    expanded, seen_edges, collected = set(), set(), []
    supernode_events = []

    while frontier and len(collected) < GLOBAL_EDGE_CAP:
        node_id, hop = frontier.popleft()
        if node_id in expanded or hop >= max_hops:
            continue
        expanded.add(node_id)

        degree = node_degree(node_id)
        limit = int(edge_limit)
        if degree > SUPER_NODE_DEGREE:
            limit = min(limit, SUPER_NODE_EDGE_CAP)
            supernode_events.append({"node_id":node_id,"degree":degree,"limit":limit})

        for e in recent_edges(node_id, limit):
            key = (e["source_id"],e["relation"],e["target_id"],e["source_chunk_id"])
            if key in seen_edges:
                continue
            seen_edges.add(key)
            collected.append(e)
            if len(collected) >= GLOBAL_EDGE_CAP:
                break

            nb = e.get("neighbor_id")
            if nb and nb not in expanded and hop + 1 < max_hops:
                frontier.append((nb, hop+1))

    out = {
        "context": textualize(collected),
        "edges": pd.DataFrame(collected),
        "diagnostics": {
            "matched_seeds": seeds,
            "expanded_nodes": len(expanded),
            "collected_edges": len(collected),
            "supernode_events": supernode_events,
        }
    }
    return out if return_debug else out["context"]

# 3.3 — Policy is tested offline separately from real-data graph measurements
print({'hops': 2, 'degree_threshold': SUPER_NODE_DEGREE,
       'supernode_edge_cap': SUPER_NODE_EDGE_CAP, 'global_edge_cap': GLOBAL_EDGE_CAP,
       'graph_context_chars': MAX_GRAPH_CONTEXT_CHARS})
print('A separate supernode fixture is checked in section 5.1.')


{'hops': 2, 'degree_threshold': 100, 'supernode_edge_cap': 50, 'global_edge_cap': 250, 'graph_context_chars': 14000}
A separate supernode fixture is checked in section 5.1.


In [15]:
#@title 3.4 — Flat answer vs Hybrid GraphRAG answer
ANSWER_SYSTEM = """
Answer only from supplied context.
Be concise but complete. Do not invent facts.
Cite provenance inline as [chunk_id=...] whenever possible.
If evidence is insufficient or conflicting, say so.
""".strip()

def generate_answer(question, context):
    prompt = f"QUESTION:\n{question}\n\nCONTEXT:\n{context}\n\nANSWER:"
    t0 = time.perf_counter()
    text, usage = groq_chat(
        [{"role":"system","content":ANSWER_SYSTEM},
         {"role":"user","content":prompt}],
        model=GROQ_MODEL
    )
    return {
        "answer": text.strip(),
        "latency_s": time.perf_counter()-t0,
        "total_tokens": usage.get("total_tokens"),
    }

def answer_flat_rag(question):
    started = time.perf_counter()
    context, retrieved = retrieve_flat_context(question, k=6)
    retrieval_latency = time.perf_counter() - started
    out = generate_answer(question, context)
    out.update(generation_latency_s=out['latency_s'], retrieval_latency_s=retrieval_latency,
               end_to_end_latency_s=time.perf_counter()-started)
    out['latency_s'] = out['end_to_end_latency_s']
    out['online_total_tokens'] = out['total_tokens']
    out.update({"context":context,"retrieved":retrieved})
    return out

def answer_graph_rag(question):
    started = time.perf_counter()
    usage_start = len(USAGE_LOG)
    g = retrieve_graph_context(question, max_hops=2, edge_limit=50, return_debug=True)
    vctx, vdocs = retrieve_flat_context(question, k=4)
    retrieval_usage = USAGE_LOG[usage_start:]
    retrieval_latency = time.perf_counter() - started
    context = f"=== GRAPH ===\n{g['context']}\n\n=== VECTOR ===\n{vctx}"
    out = generate_answer(question, context)
    seed_tokens = [entry.get('total_tokens') for entry in retrieval_usage]
    out.update(generation_latency_s=out['latency_s'], retrieval_latency_s=retrieval_latency,
               end_to_end_latency_s=time.perf_counter()-started,
               seed_extraction_tokens=sum(seed_tokens) if all(x is not None for x in seed_tokens) else None)
    out['latency_s'] = out['end_to_end_latency_s']
    out['online_total_tokens'] = (out['total_tokens'] + out['seed_extraction_tokens']
                                  if out['total_tokens'] is not None and out['seed_extraction_tokens'] is not None else None)
    out.update({"context":context,"graph_debug":g,"vector_docs":vdocs})
    return out

# 3.4 — Smoke answers. Golden references are not passed to either generator.
if retrieval_ready and llm_ready:
    smoke_question = golden_df.iloc[0].question
    smoke_answers = cached_json('smoke', {
        'version': PROMPT_VERSION, 'question': smoke_question,
        'corpus': fingerprint(chunks_df), 'model': GROQ_MODEL,
        'reasoning_effort': GROQ_REASONING_EFFORT, 'namespace': LAB_NAMESPACE},
        lambda: {'flat': answer_flat_rag(smoke_question)['answer'],
                 'graph': answer_graph_rag(smoke_question)['answer']})
    display(smoke_answers)
else:
    print('PENDING W4: real answers have not been generated.')


{'flat': 'The July expansion references the external provider **Cohere** – Amazon added “technology access from the startup Cohere” – and alongside it announced a new **health‑care system that can generate clinical notes after a patient visit**【chunk_id=613de2c24d2456048c08::c0000】.',
 'graph': 'The July expansion of Amazon’s AI service references the external technology provider **Cohere** (the startup whose technology is being used to build more conversational customer‑service agents) and, alongside that, Amazon announced a **health‑care system that can generate clinical notes after a patient visit**【chunk_id=613de2c24d2456048c08::c0000】.'}

# PHẦN 4 — GOLDEN DATASET & LLM-AS-A-JUDGE

Repo có 25 câu thực tế trong các CSV mang tên `golden_50`. Bản detailed chứa nguồn và reference answers.
Pipeline mặc định chọn 5 câu có nguồn đã xác minh: 1 factoid, 2 multi-hop, 2 cross-doc.
`outputs/golden_coverage.csv` phân biệt độ phủ corpus, vector và extraction.
Reference answers chỉ đưa cho Judge; không đưa vào retrieval, extraction hoặc generator.
Latency chính đo retrieval + generation; token online GraphRAG gồm seed extraction + generation.


In [16]:
def validate_golden(df, require_answers=True):
    required = {'id', 'group', 'question', 'reference_answer'}
    if not required <= set(df.columns):
        raise ValueError(f'Missing Golden columns: {required - set(df.columns)}')
    if len(df) < 5 or not {'factoid', 'multi-hop', 'cross-doc'} <= set(df.group):
        raise ValueError('Golden needs at least 5 questions covering all three groups.')
    if df.id.isna().any() or df.id.astype(str).str.strip().eq('').any() or df.id.duplicated().any():
        raise ValueError('Golden IDs must be nonempty and unique.')
    if df.question.fillna('').str.strip().eq('').any():
        raise ValueError('Golden question is empty.')
    if require_answers and df.reference_answer.fillna('').str.strip().eq('').any():
        raise ValueError('Golden reference_answer is empty.')
    return True

# 4.1 — Verify the selected Golden; original CSV names do not imply 50 records
original_golden = pd.read_csv(ROOT / 'data/graphrag_golden_50_first5000_detailed.csv')
print('Available Golden records:', len(original_golden))
display(original_golden.group.value_counts())
if data_ready:
    validate_golden(golden_df)
    display(golden_df[['id', 'group', 'question']])
else:
    print('PENDING W1/W5: source coverage must be verified before choosing final Golden queries.')


Available Golden records: 25


group
multi-hop    12
cross-doc    11
factoid       2
Name: count, dtype: int64

,id,group,question
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili..."
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w..."
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...
15,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ..."


In [17]:
#@title 4.2 — LLM-as-a-Judge
JUDGE_SYSTEM = """
You are a strict evaluator of RAG answers.
Score 1-5:
- comprehensiveness
- faithfulness to supplied candidate context
- multi_hop_reasoning accuracy
Use the reference answer as correctness anchor.
Return strict JSON only.
""".strip()

def judge_credentials():
    name = JUDGE_KEY_NAMES.get(JUDGE_PROVIDER)
    if name is None:
        raise ValueError('JUDGE_PROVIDER must be openai, groq or gemini.')
    return name, globals()[name]


def judge_json(system, user, purpose='judge'):
    if not JUDGE_MODEL:
        raise RuntimeError("Thiếu JUDGE_MODEL.")

    if JUDGE_PROVIDER == "groq":
        return groq_json(system, user, model=JUDGE_MODEL, purpose=purpose)[0]

    key_name, key = judge_credentials()
    if not is_configured(key):
        raise RuntimeError(f'Thiếu {key_name}.')
    from openai import OpenAI
    options = {'api_key': key, 'timeout': 60.0, 'max_retries': 3}
    response_format = {'type': 'json_object'}
    if JUDGE_PROVIDER == 'gemini':
        # Google's documented compatibility endpoint; requests use the Gemini key.
        options['base_url'] = 'https://generativelanguage.googleapis.com/v1beta/openai/'
        properties = ({'ok': {'type': 'boolean'}} if purpose == 'preflight' else {
            **{axis: {'type': 'integer', 'minimum': 1, 'maximum': 5} for axis in
               ['comprehensiveness', 'faithfulness', 'multi_hop_reasoning']},
            'rationale': {'type': 'string'},
        })
        response_format = {'type': 'json_schema', 'json_schema': {
            'name': 'lab19_judge_response', 'strict': True,
            'schema': {'type': 'object', 'properties': properties,
                       'required': list(properties), 'additionalProperties': False},
        }}
    try:
        with OpenAI(**options) as client:
            resp = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role': 'system', 'content': system}, {'role': 'user', 'content': user}],
                temperature=0.0,
                response_format=response_format,
            )
        usage = getattr(resp, 'usage', None)
        USAGE_LOG.append({'provider': JUDGE_PROVIDER, 'model': JUDGE_MODEL, 'purpose': purpose,
                          **{field: getattr(usage, field, None) for field in
                             ['prompt_tokens', 'completion_tokens', 'total_tokens']}})
        return parse_json_object(resp.choices[0].message.content)
    except Exception as error:
        raise RuntimeError(safe_error(error)) from None

def judge_answer(question, reference, answer, context):
    prompt = f"""
QUESTION:
{question}

REFERENCE:
{reference}

CANDIDATE:
{answer}

CANDIDATE CONTEXT:
{context}

Return:
{{
 "comprehensiveness":1,
 "faithfulness":1,
 "multi_hop_reasoning":1,
 "rationale":"2-5 sentences"
}}
"""
    obj = cached_json('judge', {'version': PROMPT_VERSION, 'provider': JUDGE_PROVIDER,
                               'model': JUDGE_MODEL, 'reasoning_effort': GROQ_REASONING_EFFORT,
                               'system': JUDGE_SYSTEM, 'prompt': prompt},
                      lambda: judge_json(JUDGE_SYSTEM, prompt))
    out = {}
    for k in ["comprehensiveness","faithfulness","multi_hop_reasoning"]:
        score = obj.get(k)
        if isinstance(score, bool) or not isinstance(score, int) or not 1 <= score <= 5:
            raise ValueError(f'Invalid Judge score for {k}: expected integer 1-5.')
        out[k] = score
    out["rationale"] = norm_space(obj.get("rationale"))
    if not out['rationale']:
        raise ValueError('Judge rationale is empty.')
    return out

# 4.2 — Judge uses the full candidate context and validates integer scores 1–5
print('Judge provider:', JUDGE_PROVIDER, '| Model:', JUDGE_MODEL)
print('Judge configured:', bool(JUDGE_MODEL and JUDGE_PROVIDER in JUDGE_KEY_NAMES
                                and is_configured(judge_credentials()[1])))


Judge provider: groq | Model: openai/gpt-oss-120b
Judge configured: True


In [18]:
#@title 4.3 — Evaluation runner + checkpoint
def run_evaluation(golden_df):
    validate_golden(golden_df)
    rows = []
    manifest = json.loads((OUTPUTS / 'corpus_manifest.json').read_text(encoding='utf-8'))
    evaluation_config = {'version': PROMPT_VERSION, 'corpus': manifest['chunks_fingerprint'],
                         'extraction': manifest['extraction_fingerprint'], 'golden': fingerprint(golden_df),
                         'generator': GROQ_MODEL, 'judge_provider': JUDGE_PROVIDER, 'judge': JUDGE_MODEL,
                         'reasoning_effort': GROQ_REASONING_EFFORT,
                         'graph_namespace': LAB_NAMESPACE, 'embedding': EMBED_MODEL,
                         'answer_prompt': ANSWER_SYSTEM, 'judge_prompt': JUDGE_SYSTEM,
                         'entity_threshold': 0.90, 'seed_threshold': 0.66, 'hops': 2,
                         'flat_k': 6, 'hybrid_vector_k': 4, 'edge_cap': GLOBAL_EDGE_CAP,
                         'node_cap': SUPER_NODE_EDGE_CAP, 'context_cap': MAX_GRAPH_CONTEXT_CHARS}
    run_id = fingerprint(evaluation_config)
    write_json(OUTPUTS / 'run_manifest.json', {**manifest, **evaluation_config, 'run_id': run_id,
               'latency_definition': 'retrieval plus generation wall-clock seconds, excludes Judge',
               'token_definition': 'online generator plus seed extraction; offline indexing/Judge separate'})
    existing = {}
    if CHECKPOINT.exists():
        saved = pd.read_csv(CHECKPOINT).fillna('')
        if 'run_id' in saved.columns:
            existing = {r['id']: r for r in saved[saved.run_id.eq(run_id)].to_dict('records')}
    errors = []
    for q in tqdm(golden_df.itertuples(index=False), total=len(golden_df), desc="Evaluation"):
        if q.id in existing:
            rows.append(existing[q.id])
            continue
        try:
            # Save completed answers before Judge so a Judge failure does not repeat retrieval/generation.
            def compute_answers():
                flat_answer = answer_flat_rag(q.question)
                graph_answer = answer_graph_rag(q.question)
                def serializable(out):
                    return {key: value.to_dict('records') if isinstance(value, pd.DataFrame) else
                            {k: v.to_dict('records') if isinstance(v, pd.DataFrame) else v for k, v in value.items()}
                            if isinstance(value, dict) else value for key, value in out.items()}
                return {'flat': serializable(flat_answer), 'graph': serializable(graph_answer)}
            answers = cached_json('answers', {'run_id': run_id, 'id': q.id, 'question': q.question}, compute_answers)
            flat, graph = answers['flat'], answers['graph']
            jf = judge_answer(q.question, q.reference_answer, flat['answer'], flat['context'])
            jg = judge_answer(q.question, q.reference_answer, graph['answer'], graph['context'])
        except Exception as error:
            errors.append({'id': q.id, 'error': safe_error(error)})
            pd.DataFrame(errors).to_csv(OUTPUTS / 'evaluation_errors.csv', index=False)
            print('Evaluation failed:', q.id, safe_error(error))
            continue

        rows.append({
            "id":q.id, "group":q.group, "question":q.question,
            "reference_answer":q.reference_answer,
            'run_id': run_id,
            "flat_answer":flat["answer"], "graph_answer":graph["answer"],
            "flat_comprehensiveness":jf["comprehensiveness"],
            "graph_comprehensiveness":jg["comprehensiveness"],
            "flat_faithfulness":jf["faithfulness"],
            "graph_faithfulness":jg["faithfulness"],
            "flat_multi_hop_reasoning":jf["multi_hop_reasoning"],
            "graph_multi_hop_reasoning":jg["multi_hop_reasoning"],
            "flat_latency_s":flat["latency_s"],
            "graph_latency_s":graph["latency_s"],
            "flat_total_tokens":flat.get("online_total_tokens"),
            "graph_total_tokens":graph.get("online_total_tokens"),
            'flat_generation_tokens': flat.get('total_tokens'),
            'graph_generation_tokens': graph.get('total_tokens'),
            'graph_seed_tokens': graph.get('seed_extraction_tokens'),
            'flat_retrieval_latency_s': flat['retrieval_latency_s'],
            'graph_retrieval_latency_s': graph['retrieval_latency_s'],
            'flat_generation_latency_s': flat['generation_latency_s'],
            'graph_generation_latency_s': graph['generation_latency_s'],
            "flat_judge_rationale":jf["rationale"],
            "graph_judge_rationale":jg["rationale"],
            "graph_supernode_events":len(
                graph["graph_debug"]["diagnostics"].get("supernode_events",[])
            )
        })
        pd.DataFrame(rows).to_csv(CHECKPOINT, index=False)
        write_json(OUTPUTS / '.cache' / f'trace_{run_id}_{q.id}.json', answers)
        write_json(OUTPUTS / 'usage_log.json', USAGE_LOG)
    if errors:
        raise RuntimeError(f'{len(errors)} Golden queries failed. Check evaluation_errors.csv; rerun resumes successful queries.')
    result = pd.DataFrame(rows)
    result.to_csv(OUTPUTS / 'graphrag_eval_results.csv', index=False)
    return result

# validate_golden(golden_df, require_answers=True)
# eval_results_df = run_evaluation(golden_df)
# display(eval_results_df)

# 4.3 — Resume by run fingerprint/query ID; cache answers before Judge calls
if retrieval_ready and llm_ready:
    eval_results_df = run_evaluation(golden_df)
    display(eval_results_df)
else:
    print('PENDING W5: evaluation has not run; no scores/tokens/latency are fabricated.')


Evaluation:   0%|          | 0/5 [00:00<?, ?it/s]

Evaluation: 100%|██████████| 5/5 [00:00<00:00, 2396.47it/s]

,id,group,question,reference_answer,run_id,flat_answer,graph_answer,flat_comprehensiveness,graph_comprehensiveness,flat_faithfulness,...,flat_generation_tokens,graph_generation_tokens,graph_seed_tokens,flat_retrieval_latency_s,graph_retrieval_latency_s,flat_generation_latency_s,graph_generation_latency_s,flat_judge_rationale,graph_judge_rationale,graph_supernode_events
0,G5000-26,multi-hop,"What external technology provider is named inside Amazon's July AI-service expansion, and what other new AI capabili...",Amazon's AI-service story names access to technology from Cohere. It also mentions a program for building more conve...,9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,The July expansion references the external provider **Cohere** – Amazon added “technology access from the startup Co...,The July expansion of Amazon’s AI service references the external technology provider **Cohere** — the startup whose...,5,5,5,...,813,677,316,0.066660,1.291701,6.783573,0.968049,The candidate correctly identifies Cohere as the external technology provider and the healthcare system that generat...,The candidate correctly identifies Cohere as the external technology provider and accurately mentions the new health...,0
1,G5000-27,cross-doc,How should the graph reconcile the statement that AMD powers multiple cloud services with the later Reuters report a...,The June 1 investment article broadly says AMD powers multiple cloud services through its chips. The June 14 Reuters...,9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,The graph can treat the two pieces of information as complementary rather than contradictory:\n\n* **General claim:*...,"The graph’s “AMD powers multiple cloud services” claim (from the June 1, 2023 source) describes the broader market –...",5,5,5,...,1023,821,279,0.335124,1.375014,0.982089,1.141286,The answer restates the broad claim that AMD powers multiple cloud services and correctly notes the Reuters report t...,The answer restates the broad June 1 claim that AMD powers multiple cloud services and correctly contrasts it with t...,0
2,G5000-28,multi-hop,"Which model providers are connected to Google Cloud Next '23 in the selected data, and which models are associated w...",Meta is connected via Llama 2 and Code Llama; the Technology Innovation Institute is connected via Falcon LLM; Anthr...,9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,- **Meta** – Llama 2 and Code Llama [chunk_id=7ca725290aab95bd56a7] \n- **Technology Innovative Institute** – Falco...,- **Meta** – provides **Llama 2** and **Code Llama**【chunk_id=7ca725290aab95bd56a7::c0000】 \n- **Technology Innovat...,5,5,5,...,897,872,437,0.546395,2.562226,1.116739,1.214025,"The candidate lists all three model providers mentioned in the reference (Meta, Technology Innovation Institute, Ant...","The candidate correctly identifies all three model providers (Meta, Technology Innovation Institute, Anthropic) ment...",0
3,G5000-29,cross-doc,How did participation in White House AI commitments broaden from July to September 2023 according to the selected re...,"The July report names seven companies including Google, Meta, and OpenAI as making voluntary AI commitments. The Sep...",9ebf9bec11e988923beb337d91ab7ba120acef31b16fd694599b50d4e7d549af,"The July 2023 report notes that **seven** technology firms – among them Google, Meta and OpenAI – had voluntarily si...","In July 2023 the White House AI‑safety pledge was signed by a core group of **seven** firms – notably Google, Meta a...",5,5,5,...,908,988,314,0.030217,6.067750,0.802907,3.532002,"The answer accurately captures the July report's seven companies and the September report's expanded list of IBM, Ad...","The answer correctly lists the July report’s seven companies (Google, Meta, OpenAI) and the September report’s addit...",0
4,G5000-41,factoid,"Which company did HPE agree to acquire to expand edge-to-cloud security, and what unified security architecture did ...",Axis

In [19]:
#@title 4.4 — Comparison table + export
def comparison_table(eval_df):
    metric_map = {
        "Comprehensiveness":("flat_comprehensiveness","graph_comprehensiveness"),
        "Faithfulness":("flat_faithfulness","graph_faithfulness"),
        "Multi-hop reasoning":("flat_multi_hop_reasoning","graph_multi_hop_reasoning"),
        "Latency (s)":("flat_latency_s","graph_latency_s"),
        "Token usage":("flat_total_tokens","graph_total_tokens"),
    }

    rows = []
    groups = list(eval_df.groupby('group')) + [('ALL', eval_df)]
    for group, g in groups:
        for metric, (fc,gc) in metric_map.items():
            f = pd.to_numeric(g[fc], errors="coerce").mean()
            gr = pd.to_numeric(g[gc], errors="coerce").mean()

            if metric in {"Latency (s)","Token usage"}:
                comment = "Flat RAG thường rẻ/nhanh hơn." if f < gr else "GraphRAG không đắt hơn trong sample này."
            else:
                delta = gr - f
                if delta >= .75:
                    comment = "GraphRAG cải thiện rõ; kiểm tra rationale và provenance."
                elif delta <= -.5:
                    comment = "Flat RAG tốt hơn; graph extraction/retrieval có thể gây mất thông tin hoặc nhiễu."
                else:
                    comment = "Hai phương pháp gần nhau."

            rows.append({
                "Loại câu hỏi":group, "Metric":metric,
                "Flat RAG":round(f,3) if pd.notna(f) else np.nan,
                "GraphRAG":round(gr,3) if pd.notna(gr) else np.nan,
                'Delta Graph - Flat': round(gr - f, 3) if pd.notna(f) and pd.notna(gr) else np.nan,
                'Sample count': len(g),
                "Nhận xét phân tích":comment
            })
    return pd.DataFrame(rows)

# comparison_df = comparison_table(eval_results_df)
# display(comparison_df)
# eval_results_df.to_csv("/content/graphrag_eval_results.csv", index=False)
# comparison_df.to_csv("/content/graphrag_vs_flatrag_summary.csv", index=False)

if eval_results_df is not None:
    comparison_df = comparison_table(eval_results_df)
    comparison_df.to_csv(OUTPUTS / 'graphrag_vs_flatrag_summary.csv', index=False)
    display(comparison_df)
    print('Required CSVs saved in outputs/.')
    print('Reports: lab_report.md, technical_defense.md, failure_analysis.md, reflection_VuDuyDiep.md.')
    print('When changing a benchmark run, review and update the report conclusions against the new answers.')
else:
    raise RuntimeError('Evaluation did not complete; no report metrics were invented.')


,Loại câu hỏi,Metric,Flat RAG,GraphRAG,Delta Graph - Flat,Sample count,Nhận xét phân tích
0,cross-doc,Comprehensiveness,5.000,5.000,0.000,2,Hai phương pháp gần nhau.
1,cross-doc,Faithfulness,5.000,5.000,0.000,2,Hai phương pháp gần nhau.
2,cross-doc,Multi-hop reasoning,5.000,5.000,0.000,2,Hai phương pháp gần nhau.
3,cross-doc,Latency (s),1.075,6.058,4.983,2,Flat RAG thường rẻ/nhanh hơn.
4,cross-doc,Token usage,965.500,1201.000,235.500,2,Flat RAG thường rẻ/nhanh hơn.
5,factoid,Comprehensiveness,5.000,5.000,0.000,1,Hai phương pháp gần nhau.
6,factoid,Faithfulness,5.000,5.000,0.000,1,Hai phương pháp gần nhau.
7,factoid,Multi-hop reasoning,5.000,5.000,0.000,1,Hai phương pháp gần nhau.
8,factoid,Latency (s),6.908,10.152,3.244,1,Flat RAG thường rẻ/nhanh hơn.
9,factoid,Token usage,854.000,1195.000,341.000,1,Flat RAG thường rẻ/nhanh hơn.


Required CSVs saved in outputs/.
Reports: lab_report.md, technical_defense.md, failure_analysis.md, reflection_VuDuyDiep.md.
When changing a benchmark run, review and update the report conclusions against the new answers.


# PHẦN 5 — FAILURE-MODE CHECKS & SUBMISSION (105–120')

Bắt buộc chứng minh:
1. Edge provenance không thiếu.
2. Entity Resolution có audit.
3. Super-node degree > 100 chỉ expand tối đa 50 edge.
4. Có comparison table.

In [20]:
#@title 5.1 — Super-node check + entity audit
def test_supernode_policy():
    rows = run_cypher("""
    MATCH (n:Lab19 {lab_namespace:$namespace})-[r]-()
    WHERE r.lab_namespace=$namespace
    WITH n, count(r) AS degree
    ORDER BY degree DESC LIMIT 1
    RETURN n.id AS id, n.name AS name, degree
    """)
    if not rows:
        print("Graph empty.")
        return {'status': 'not_exercised', 'reason': 'empty_graph'}

    n = rows[0]
    limit = 50 if n["degree"] > SUPER_NODE_DEGREE else 1000
    edges = recent_edges(n["id"], limit)
    print(n, "fetched=", len(edges))
    if n["degree"] > SUPER_NODE_DEGREE:
        assert len(edges) <= 50
        print('Super-node cap OK.')
    return {'status': 'passed' if n['degree'] > SUPER_NODE_DEGREE else 'not_exercised',
            'reason': '' if n['degree'] > SUPER_NODE_DEGREE else 'no_real_supernode',
            'degree': n['degree'], 'fetched': len(edges)}

def show_resolution_audit(audit_df):
    if audit_df.empty:
        print("No audit rows.")
        return
    display(
        audit_df.sort_values("similarity", ascending=False).head(30)
    )
    print('High-similarity rejected pairs:')
    display(audit_df[audit_df.decision.eq('REJECT_GUARD')].sort_values('similarity', ascending=False).head(20))







# test_supernode_policy()
# show_resolution_audit(entity_resolution_audit_df)

def verify_supernode_fixture():
    import uuid
    from datetime import timedelta
    global LAB_NAMESPACE, match_seeds
    before = run_cypher('MATCH (n) RETURN count(n) AS n')[0]['n']
    original_namespace, original_match_seeds = LAB_NAMESPACE, match_seeds
    LAB_NAMESPACE = 'day19-test-fixture-' + uuid.uuid4().hex
    hub_id = LAB_NAMESPACE + ':hub'
    nodes = [{'id': hub_id, 'name': 'LAB19 TEST HUB', 'name_norm': 'lab19 test hub',
              'type': 'Company', 'aliases': [], 'aliases_norm': []}]
    triples = []
    for i in range(120):
        target_id = f'{LAB_NAMESPACE}:target:{i}'
        nodes.append({'id': target_id, 'name': f'LAB19 TEST TECHNOLOGY {i}',
                      'name_norm': f'lab19 test technology {i}', 'type': 'Technology',
                      'aliases': [], 'aliases_norm': []})
        triples.append({'source_id': hub_id, 'target_id': target_id, 'relation': 'USES',
                        'source_chunk_id': f'fixture::{i}', 'published_date': (datetime(2023, 1, 1) + timedelta(days=i)).date().isoformat(),
                        'evidence': 'Explicit synthetic fixture for integration testing, not a corpus fact.',
                        'confidence': 1.0})
    result = {'data_kind': 'synthetic fixture in a real local Neo4j connection', 'status': 'failed'}
    try:
        nodes_df, triples_df = pd.DataFrame(nodes), pd.DataFrame(triples)
        bulk_insert_nodes(nodes_df)
        bulk_insert_edges(triples_df)
        bulk_insert_nodes(nodes_df)
        bulk_insert_edges(triples_df)
        count = run_cypher('MATCH ()-[r]->() WHERE r.lab_namespace=$namespace RETURN count(r) AS n')[0]['n']
        assert count == 120, f'Rerun duplicated edges: {count}'
        assert node_degree(hub_id) == 120
        match_seeds = lambda question: [{'id': hub_id, 'name': 'LAB19 TEST HUB', 'type': 'Company'}]
        graph = retrieve_graph_context('Integration fixture', max_hops=2, edge_limit=1000, return_debug=True)
        assert len(graph['edges']) == 50
        expected_dates = {r['published_date'] for r in triples[-50:]}
        assert set(graph['edges'].published_date) == expected_dates
        assert len(graph['context']) <= MAX_GRAPH_CONTEXT_CHARS
        result.update(status='passed', idempotent_edges=count, hub_degree=120,
                      retrieved_edges=len(graph['edges']), retained_newest_50=True,
                      context_chars=len(graph['context']), checked_at_utc=datetime.now(timezone.utc).isoformat())
    finally:
        # Only nodes created under the unique namespace above are removed.
        run_cypher('MATCH (n:Lab19 {lab_namespace:$namespace}) DETACH DELETE n')
        after = run_cypher('MATCH (n) RETURN count(n) AS n')[0]['n']
        result.update(nodes_before=before, nodes_after_cleanup=after, fixture_cleaned=(before == after))
        write_json(OUTPUTS / 'neo4j_fixture_verification.json', result)
        LAB_NAMESPACE, match_seeds = original_namespace, original_match_seeds
    assert after == before, 'Fixture cleanup changed existing data.'
    return result

if graph_ready:
    supernode_status = test_supernode_policy()
    show_resolution_audit(entity_resolution_audit_df)
    display(verify_supernode_fixture())

    # Separate synthetic name pairs, measured with the real local embedding model.
    guard_rows = []
    for left, right in [('Microsoft Windows 11', 'Microsoft Windows 11 Pro'),
                        ('Microsoft Windows 11 Pro', 'Microsoft Windows 11 Pro N')]:
        vectors = get_embedder().encode([left, right], normalize_embeddings=True, show_progress_bar=False)
        similarity = float(vectors[0] @ vectors[1])
        guard = merge_guard(left, right, typ='Technology')
        guard_rows.append({'data_kind': 'synthetic fixture, not corpus ER audit', 'left': left, 'right': right,
                           'similarity': similarity, 'guard_passed': guard,
                           'decision': 'REJECT_GUARD' if similarity >= .90 and not guard else 'REJECT_THRESHOLD'})
    display(pd.DataFrame(guard_rows))

    # Inspect original and rewritten text, including the real Samsung/Aqara mistake.
    spotcheck_notes = {'dcaa0ba4dc9d6a046c47::c0000': 'Hợp lý: its -> Synopsys; foundry -> TSMC theo câu/title trong cùng chunk.', '69178d0227a0ff6c4b24::c0000': 'Sai: the company bị thay bằng sản phẩm PolsoTM khi tên công ty chưa có trong snippet. Không có triple được chấp nhận từ chunk này.', '82639690c6f0f9db3de3::c0000': 'Guard reject do 144366 đổi thành 144,366. Giá trị số không đổi; đây là false positive vì so sánh token số theo chuỗi.', '809b337a488d409aeebb::c0000': 'Hợp lý: he -> Mike Dixon, được nêu ngay trước đó.', '8e74089254231622e67f::c0000': 'Sai: smart home manufacturer là Aqara; rewrite thành Samsung working with Samsung. Triple cuối vẫn Samsung–Aqara nhờ evidence từ title gốc.'}
    coref_review = pd.read_csv(OUTPUTS / 'coref_audit.csv').fillna('')
    coref_review = coref_review[coref_review.chunk_id.isin(spotcheck_notes)].copy()
    coref_review['review_note'] = coref_review.chunk_id.map(spotcheck_notes)
    display(coref_review)

    # Expose the actual benchmark contexts without adding submission files.
    for row in eval_results_df[eval_results_df.id.isin(['G5000-29', 'G5000-26'])].itertuples(index=False):
        trace = json.loads((CACHE / f'trace_{row.run_id}_{row.id}.json').read_text(encoding='utf-8'))
        print('Failure review:', row.id)
        print('Flat context:', trace['flat']['context'])
        print('Graph diagnostics:', trace['graph']['graph_debug']['diagnostics'])
        print('Graph edges:', trace['graph']['graph_debug']['edges'])

    recorded_diagnostic = {'label': 'Recorded earlier diagnostic, excluded from final CSV summary', 'run_id': '2b7b5ef7a40b3cbfb9a86bc063936f6a7cdca560a13caf698dcf6f8bbc62fca9', 'id': 'G5000-29', 'question': 'How did participation in White House AI commitments broaden from July to September 2023 according to the selected reports?', 'flat_answer': 'In July\u202f2023 the White House’s first AI‑safety pledge was signed by **seven** technology firms (e.g., Google, Meta, OpenAI)\u202f[chunk_id=9781172d2260895c1d09::c0000]. By mid‑September\u202f2023 the pledge had expanded to include **IBM, Adobe, Salesforce and five additional companies**, bringing the total to **eight** signatories\u202f[chunk_id=8d2f8b148829f4949b97::c0000]. Thus participation grew from seven firms in July to eight firms in September\u202f2023.', 'graph_answer': 'In July\u202f2023 the White\u202fHouse AI‑safety pledge was signed by a core group of **seven** firms – notably Google, Meta and OpenAI (among others)\u202f【chunk_id=9781172d2260895c1d09::c0000】.  \n\nBy mid‑September\u202f2023 the roster had expanded: **IBM, Adobe, Salesforce and five additional technology companies** had also joined the voluntary safety, security and transparency commitments\u202f【chunk_id=8d2f8b148829f4949b97::c0000】.  \n\nThus, participation broadened from the original seven companies in July to at least twelve (the original seven plus IBM, Adobe, Salesforce and five newcomers) by September\u202f2023.', 'judges': {'flat': {'comprehensiveness': 1, 'faithfulness': 1, 'multi_hop_reasoning': 1, 'rationale': 'The answer misstates the September total as eight signatories, contradicting the reference which indicates IBM, Adobe, Salesforce plus five additional companies (implying more than seven). It fails to capture the full expansion and misrepresents the numbers, showing poor faithfulness and inadequate multi‐hop reasoning.'}, 'graph': {'comprehensiveness': 5, 'faithfulness': 5, 'multi_hop_reasoning': 5, 'rationale': 'The answer correctly identifies the July report’s seven companies (including Google, Meta, OpenAI) and the September report’s addition of IBM, Adobe, Salesforce and five others, showing the expansion from seven to at least twelve firms. It faithfully reflects the reference details and requires only a straightforward link between the two reports, which it handles accurately.'}}}
    display(recorded_diagnostic)


{'id': '18a2f0e496fb60f2b80e8775', 'name': 'Google Cloud', 'degree': 3} fetched= 3


,type,left,right,similarity,decision,reason,canonical_left,canonical_right
39,Company,Amazon Publisher Services,Amazon Web Services,0.683461,REJECT_THRESHOLD,similarity_below_threshold,Amazon Publisher Services,Amazon Web Services
0,Company,Amazon,Amazon Web Services,0.646989,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Web Services
169,Company,Synopsys,Synechron,0.618264,REJECT_THRESHOLD,similarity_below_threshold,Synopsys,Synechron
131,Company,Microsoft,Salesforce,0.598405,REJECT_THRESHOLD,similarity_below_threshold,Microsoft,Salesforce
1,Company,Amazon,Amazon Publisher Services,0.585162,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Amazon Publisher Services
248,Technology,AI-Powered Personal Cloud Platform,AI-Powered Customer Service Suite,0.575279,REJECT_THRESHOLD,similarity_below_threshold,AI-Powered Personal Cloud Platform,AI-Powered Customer Service Suite
204,Company,Apple,Samsung,0.575256,REJECT_THRESHOLD,similarity_below_threshold,Apple,Samsung
87,Company,IT World Canada,Worldpay,0.573572,REJECT_THRESHOLD,similarity_below_threshold,IT World Canada,Worldpay
2,Company,Amazon,Salesforce,0.537856,REJECT_THRESHOLD,similarity_below_threshold,Amazon,Salesforce
36,Company,IAS,IHSE USA,0.520189,REJECT_THRESHOLD,similarity_below_threshold,IAS,IHSE USA


High-similarity rejected pairs:


,type,left,right,similarity,decision,reason,canonical_left,canonical_right


{'data_kind': 'synthetic fixture in a real local Neo4j connection',
 'status': 'passed',
 'idempotent_edges': 120,
 'hub_degree': 120,
 'retrieved_edges': 50,
 'retained_newest_50': True,
 'context_chars': 9589,
 'checked_at_utc': '2026-10-05T16:09:32.924587+00:00',
 'nodes_before': 194,
 'nodes_after_cleanup': 194,
 'fixture_cleaned': True}

,data_kind,left,right,similarity,guard_passed,decision
0,"synthetic fixture, not corpus ER audit",Microsoft Windows 11,Microsoft Windows 11 Pro,0.896877,False,REJECT_THRESHOLD
1,"synthetic fixture, not corpus ER audit",Microsoft Windows 11 Pro,Microsoft Windows 11 Pro N,0.949005,False,REJECT_GUARD


,chunk_id,resolved_text,unresolved_mentions,coref_status,original_text,review_note
85,dcaa0ba4dc9d6a046c47::c0000,Synopsys TSMC team up to help clients accelerate 2nm chips designs. EDA specialist Synopsys has recently announced S...,[],resolved,Synopsys TSMC team up to help clients accelerate 2nm chips designs. EDA specialist Synopsys has recently announced i...,Hợp lý: its -> Synopsys; foundry -> TSMC theo câu/title trong cùng chunk.
96,69178d0227a0ff6c4b24::c0000,Promising Remote Patient Monitoring solution PolsoTM Achieves 1st FDA Clearance. PolsoTM PolsoTM Remote Patient Moni...,[],resolved,Promising Remote Patient Monitoring solution PolsoTM Achieves 1st FDA Clearance. PolsoTM the company''s Remote Patie...,Sai: the company bị thay bằng sản phẩm PolsoTM khi tên công ty chưa có trong snippet. Không có triple được chấp nhận...
159,82639690c6f0f9db3de3::c0000,Top 10 technology and ethics stories of 2022. when use of the technology was paused as a result of the pandemic. Whi...,"[""COREF_CHANGED_NUMBERS""]",rejected,Top 10 technology and ethics stories of 2022. when use of the technology was paused as a result of the pandemic. Whi...,"Guard reject do 144366 đổi thành 144,366. Giá trị số không đổi; đây là false positive vì so sánh token số theo chuỗi."
193,809b337a488d409aeebb::c0000,Focusing On Growth: Synergy Sustainability And Customer Loyalty In Foodservice. I recently spoke with Focus Brands’ ...,[],resolved,Focusing On Growth: Synergy Sustainability And Customer Loyalty In Foodservice. I recently spoke with Focus Brands’ ...,"Hợp lý: he -> Mike Dixon, được nêu ngay trước đó."
194,8e74089254231622e67f::c0000,SDC23: Samsung and Aqara Partners to Demonstrate Presence Sensor FP2 on SmartThings Platform. The FP2 Presence Senso...,[],resolved,SDC23: Samsung and Aqara Partners to Demonstrate Presence Sensor FP2 on SmartThings Platform. The FP2 Presence Senso...,Sai: smart home manufacturer là Aqara; rewrite thành Samsung working with Samsung. Triple cuối vẫn Samsung–Aqara nhờ...


Failure review: G5000-26
Flat context: [chunk_id=0d53dc66fb96e3cb91e2::c0000 | date=2023-06-14 | score=0.733]
Exclusive: Amazon's cloud unit is considering AMD's new AI chips. Amazon Web Services the world''s largest cloud computing provider is considering using new artificial intelligence chips from Advanced Micro Devices Inc though it has not made a final decision an AWS executive told Reuters.

[chunk_id=613de2c24d2456048c08::c0000 | date=2023-07-26 | score=0.723]
Exclusive: Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announced new AI tools on Wednesday including a program for building more conversational customer-service agents technology access from the startup Cohere and a healthcare ...

[chunk_id=ae3e878731488df141c0::c0000 | date=2023-07-27 | score=0.711]
Amazon has drawn thousands to try its AI service competing with Microsoft Google. The company also announced new AI tools on Wednesday including a program for building mo

{'label': 'Recorded earlier diagnostic, excluded from final CSV summary',
 'run_id': '2b7b5ef7a40b3cbfb9a86bc063936f6a7cdca560a13caf698dcf6f8bbc62fca9',
 'id': 'G5000-29',
 'question': 'How did participation in White House AI commitments broaden from July to September 2023 according to the selected reports?',
 'flat_answer': 'In July\u202f2023 the White House’s first AI‑safety pledge was signed by **seven** technology firms (e.g., Google, Meta, OpenAI)\u202f[chunk_id=9781172d2260895c1d09::c0000]. By mid‑September\u202f2023 the pledge had expanded to include **IBM, Adobe, Salesforce and five additional companies**, bringing the total to **eight** signatories\u202f[chunk_id=8d2f8b148829f4949b97::c0000]. Thus participation grew from seven firms in July to eight firms in September\u202f2023.',
 'graph_answer': 'In July\u202f2023 the White\u202fHouse AI‑safety pledge was signed by a core group of **seven** firms – notably Google, Meta and OpenAI (among others)\u202f【chunk_id=9781172d2260895

## 5.2 — Thuyết minh kỹ thuật: học viên tự điền

1. Coreference sai ở tình huống nào?
2. Entity threshold bao nhiêu, vì sao?
3. Candidate nào similarity cao nhưng không nên merge?
4. Top 3 super-node và degree?
5. Vì sao ưu tiên edge mới nhất có thể đúng/sai?
6. Flat RAG thắng nhóm nào?
7. GraphRAG thắng nhóm nào?
8. Latency/token trade-off?
9. AI Coding Agent đề xuất gì mà bạn **không dùng**, vì sao?
10. Scale 350MB: bottleneck đầu tiên là gì?

# 🎁 BONUS

## A — Low-level / High-level
Tạo local entities và high-level topics/community reports; query router chọn tầng retrieval.

## B — Global Search via Community Reports
Nếu Neo4j instance không có GDS phù hợp, fallback:
1. export edges,
2. NetworkX community detection,
3. `UNWIND` write `community_id`,
4. LLM summarize community,
5. query global trên reports.

## C — Self-Correction Graph Retrieval
- hop 2 → LLM kiểm tra context đủ chưa,
- thiếu → hop 3,
- vẫn thiếu → vector fallback,
- bắt buộc stop condition.

In [21]:
#@title Bonus — NetworkX community fallback
import networkx as nx

def build_communities(limit_edges=20000):
    edge_df = pd.DataFrame(run_cypher("""
    MATCH (a:Lab19 {lab_namespace:$namespace})-[r]->(b:Lab19 {lab_namespace:$namespace})
    WHERE r.lab_namespace=$namespace
    RETURN a.id AS source, b.id AS target
    LIMIT $limit
    """, limit=int(limit_edges)))

    G = nx.Graph()
    G.add_edges_from(edge_df[["source","target"]].itertuples(index=False, name=None))
    communities = nx.algorithms.community.greedy_modularity_communities(G)

    rows = []
    for cid, members in enumerate(communities):
        rows += [{"id":node_id,"community_id":int(cid)} for node_id in members]

    for b in batches(rows, 1000):
        run_cypher("""
        UNWIND $rows AS row
        MATCH (n:Lab19 {id:row.id, lab_namespace:$namespace})
        SET n.community_id=row.community_id
        """, rows=b)

    return pd.DataFrame(rows)

# community_df = build_communities()

# Bonus — Community ID alone does not meet the global-search bonus requirements
print('Optional scaffold only. Complete community reports/querying and quantify before claiming bonus.')


Optional scaffold only. Complete community reports/querying and quantify before claiming bonus.


In [22]:
#@title Bonus — Self-correction scaffold
SUFFICIENCY_SYSTEM = """
Decide whether the supplied retrieval context is sufficient to answer the question faithfully.
Do not answer the question. Return strict JSON only.
""".strip()

def context_sufficient(question, context):
    obj, _ = groq_json(
        SUFFICIENCY_SYSTEM,
        f"""QUESTION: {question}
CONTEXT:
{context[:16000]}
Return {{"sufficient":true,"missing":"..."}}"""
        , purpose='context_sufficiency'
    )
    return bool(obj.get("sufficient")), norm_space(obj.get("missing"))

def self_correcting_context(question):
    g2 = retrieve_graph_context(question, 2, 50, True)
    ok, missing = context_sufficient(question, g2["context"])
    if ok:
        return {"route":"hop2","context":g2["context"],"missing":""}

    g3 = retrieve_graph_context(question, 3, 50, True)
    ok, missing2 = context_sufficient(question, g3["context"])
    if ok:
        return {"route":"hop3","context":g3["context"],"missing":missing}

    flat, _ = retrieve_flat_context(question, k=8)
    return {
        "route":"hop3+vector",
        "context":f"=== GRAPH ===\n{g3['context']}\n\n=== VECTOR ===\n{flat}",
        "missing":missing2
    }

# Bonus — Bounded hop2 → hop3 → vector fallback; enable after the core benchmark
print('Optional scaffold only. Compare quality/latency/tokens before claiming bonus.')


Optional scaffold only. Compare quality/latency/tokens before claiming bonus.


# Rubric và bài nộp

Theo RUBRIC.md: Pipeline 40đ, Failure modes 20đ, Evaluation 20đ, Technical defense/Reflection 20đ.
Bài nộp gồm notebook này, `outputs/graphrag_eval_results.csv`, `outputs/graphrag_vs_flatrag_summary.csv`
và `reports/lab_report.md`, `reports/technical_defense.md`, `reports/failure_analysis.md`, `reports/reflection_VuDuyDiep.md`.
Hai CSV là kết quả API thật đã đo; cache replay giữ latency của request gốc, không tính thời gian đọc cache làm latency benchmark.
Review vẫn phát hiện giới hạn Judge và lỗi seed; xem báo cáo và section 5.1. Các bonus hiện chỉ là scaffold.
